# NLP Project

This notebook is the code associated with our work for the MVA Course "Algorithms for speech and language processing". In this notebook we reimplement and extend the work from "Augmentation Invariant Discrete Representation for
Generative Spoken Language Modeling" by Itai Gat et al.
The nodebook is divided in the following parts :

- Encoder extraction : Adrien Letellier
- Quantizer training & testing - all the basics : Raphaël Bernas
- Augmentation : Maxime Corlay
- Quantizers outputs for UED distance computation : Emilie Zheng
- Loading and testing new models : Maxime Corlay
- Dynamical Time Warping : Adrien Letellier
- Quantizer architecture : Raphaël Bernas

In [4]:
!pip install --pre torch torchvision torchaudio
!pip install numpy
!pip install transformers
!pip install datasets
!pip install scikit-learn
!pip install librosa
!pip install soundfile
!pip install python-Levenshtein
!pip install dtaidistance


In [5]:
import torch
from transformers import Wav2Vec2FeatureExtractor, HubertModel, Wav2Vec2Model, WavLMModel
from datasets import load_dataset
import numpy as np
from sklearn.cluster import KMeans

In [6]:
# C'est un dataset similaire à celui qu'on veut mais plus petit
# Tous les datasets sont sur HuggingFace de toute façon donc c'est facile à changer
dataset = load_dataset("hf-internal-testing/librispeech_asr_demo", "clean", split="validation", trust_remote_code=True)
dataset = dataset.sort("id")
sampling_rate = dataset.features["audio"].sampling_rate

## 1. Encoder extraction

### 1.1 HuBERT

#### 1.1.1 Speech Encoder

In [4]:
# Load feature extractor and HuBERT model
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained("facebook/hubert-base-ls960")
model = HubertModel.from_pretrained("facebook/hubert-base-ls960", output_hidden_states=True)
model.eval()

preprocessor_config.json:   0%|          | 0.00/213 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

HubertModel(
  (feature_extractor): HubertFeatureEncoder(
    (conv_layers): ModuleList(
      (0): HubertGroupNormConvLayer(
        (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,), bias=False)
        (activation): GELUActivation()
        (layer_norm): GroupNorm(512, 512, eps=1e-05, affine=True)
      )
      (1-4): 4 x HubertNoLayerNormConvLayer(
        (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
        (activation): GELUActivation()
      )
      (5-6): 2 x HubertNoLayerNormConvLayer(
        (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
        (activation): GELUActivation()
      )
    )
  )
  (feature_projection): HubertFeatureProjection(
    (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (projection): Linear(in_features=512, out_features=768, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): HubertEncoder(
    (pos_conv_embed): HubertPositionalConvEmbedding(
      (conv): Para

In [5]:
# Select first point of the dataset
input_values = feature_extractor(dataset[0]["audio"]["array"], sampling_rate=sampling_rate, return_tensors="pt")['input_values']

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

In [6]:
with torch.no_grad():
    outputs = model(input_values)

In [7]:
len(outputs['hidden_states'])

13

In [8]:
outputs['last_hidden_state']

tensor([[[ 0.0924, -0.0873,  0.2480,  ..., -0.0481,  0.1011, -0.3813],
         [ 0.1171, -0.0870,  0.2565,  ..., -0.0525,  0.0991, -0.4402],
         [ 0.1896, -0.0639,  0.2879,  ..., -0.0714,  0.0727, -0.5391],
         ...,
         [ 0.1721,  0.3426,  0.0415,  ..., -0.0303, -0.1977, -0.6863],
         [ 0.1121,  0.1157,  0.1866,  ..., -0.1068, -0.1563, -0.5571],
         [ 0.0897,  0.0344,  0.2302,  ..., -0.0846, -0.0011, -0.4501]]])

In [9]:
outputs['hidden_states'][-1]

tensor([[[ 0.0924, -0.0873,  0.2480,  ..., -0.0481,  0.1011, -0.3813],
         [ 0.1171, -0.0870,  0.2565,  ..., -0.0525,  0.0991, -0.4402],
         [ 0.1896, -0.0639,  0.2879,  ..., -0.0714,  0.0727, -0.5391],
         ...,
         [ 0.1721,  0.3426,  0.0415,  ..., -0.0303, -0.1977, -0.6863],
         [ 0.1121,  0.1157,  0.1866,  ..., -0.1068, -0.1563, -0.5571],
         [ 0.0897,  0.0344,  0.2302,  ..., -0.0846, -0.0011, -0.4501]]])

In [10]:
# Extract encoder output (this includes feature extraction)
encoder_output = outputs['last_hidden_state']  # Shape: (1, seq_len, feature_dim)
print("Encoder Output Shape:", encoder_output.shape)

Encoder Output Shape: torch.Size([1, 292, 768])


In [11]:
encoder_output

tensor([[[ 0.0924, -0.0873,  0.2480,  ..., -0.0481,  0.1011, -0.3813],
         [ 0.1171, -0.0870,  0.2565,  ..., -0.0525,  0.0991, -0.4402],
         [ 0.1896, -0.0639,  0.2879,  ..., -0.0714,  0.0727, -0.5391],
         ...,
         [ 0.1721,  0.3426,  0.0415,  ..., -0.0303, -0.1977, -0.6863],
         [ 0.1121,  0.1157,  0.1866,  ..., -0.1068, -0.1563, -0.5571],
         [ 0.0897,  0.0344,  0.2302,  ..., -0.0846, -0.0011, -0.4501]]])

#### 1.1.2 Quantizer

In [12]:
# Apply K-Means clustering
num_clusters = 50
features = encoder_output.squeeze(0).numpy()
features.shape  # Shape: (seq_len, feature_dim)

(292, 768)

In [13]:
kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
kmeans.fit(features)

KMeans(n_clusters=50, n_init=10, random_state=42)

In [14]:
# Convert encoder output to discrete representations
quantized_ids = kmeans.predict(features)
print("Discrete Representation (First 20 IDs):", quantized_ids[:20])

Discrete Representation (First 20 IDs): [33 33 33 33  9  9  9  9  9  9  9  9  9  4  4  4  4  4  4  4]


In [15]:
print(len(quantized_ids))

292


### 1.2 wav2vec2

#### 1.2.1 Speech Encoder

In [16]:
# Load feature extractor and model
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained("facebook/wav2vec2-base")
model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base")
model.eval()

preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.84k [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/configuration_utils.py:315: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/380M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/380M [00:00<?, ?B/s]

Wav2Vec2Model(
  (feature_extractor): Wav2Vec2FeatureEncoder(
    (conv_layers): ModuleList(
      (0): Wav2Vec2GroupNormConvLayer(
        (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,), bias=False)
        (activation): GELUActivation()
        (layer_norm): GroupNorm(512, 512, eps=1e-05, affine=True)
      )
      (1-4): 4 x Wav2Vec2NoLayerNormConvLayer(
        (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
        (activation): GELUActivation()
      )
      (5-6): 2 x Wav2Vec2NoLayerNormConvLayer(
        (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
        (activation): GELUActivation()
      )
    )
  )
  (feature_projection): Wav2Vec2FeatureProjection(
    (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (projection): Linear(in_features=512, out_features=768, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): Wav2Vec2Encoder(
    (pos_conv_embed): Wav2Vec2PositionalConvEmbedding(
  

In [17]:
# Select first point of the dataset
input_values = feature_extractor(dataset[0]["audio"]["array"], sampling_rate=sampling_rate, return_tensors="pt")['input_values']

In [18]:
with torch.no_grad():
    outputs = model(input_values)

In [19]:
# Extract encoder output (this includes feature extraction)
encoder_output = outputs['last_hidden_state']  # Shape: (1, seq_len, feature_dim)
print("Encoder Output Shape:", encoder_output.shape)

Encoder Output Shape: torch.Size([1, 292, 768])


In [20]:
encoder_output

tensor([[[ 0.0252, -0.0161,  0.1962,  ...,  0.5132,  0.2121, -0.1114],
         [-0.3064, -0.0877,  0.0485,  ...,  0.2346,  0.6384, -0.3538],
         [ 0.2099,  0.1193,  0.5077,  ...,  0.0555,  0.3368,  0.2325],
         ...,
         [-0.3104, -0.0688,  0.0304,  ...,  0.1952,  0.6314, -0.3537],
         [-0.3162, -0.0806,  0.0095,  ...,  0.1865,  0.6372, -0.3541],
         [-0.0199, -0.0527,  0.0903,  ...,  0.3927,  0.2868, -0.3365]]])

#### 1.2.2 Quantizer

In [21]:
# Apply K-Means clustering
num_clusters = 50
features = encoder_output.squeeze(0).numpy()
features.shape  # Shape: (seq_len, feature_dim)

(292, 768)

In [22]:
kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
kmeans.fit(features)

KMeans(n_clusters=50, n_init=10, random_state=42)

In [23]:
# Convert encoder output to discrete representations
quantized_ids = kmeans.predict(features)
print("Discrete Representation (First 20 IDs):", quantized_ids[:20])

Discrete Representation (First 20 IDs): [29  0  1  1  0  1  0  1  1  1  0  0  0  1 25 10 10 10 41  1]


### 1.3 WavLM

#### 1.3.1 Speech Encoder

In [14]:
# Load feature extractor and model
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained("microsoft/wavlm-base")
model = WavLMModel.from_pretrained("microsoft/wavlm-base")
model.eval()

WavLMModel(
  (feature_extractor): WavLMFeatureEncoder(
    (conv_layers): ModuleList(
      (0): WavLMGroupNormConvLayer(
        (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,), bias=False)
        (activation): GELUActivation()
        (layer_norm): GroupNorm(512, 512, eps=1e-05, affine=True)
      )
      (1-4): 4 x WavLMNoLayerNormConvLayer(
        (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
        (activation): GELUActivation()
      )
      (5-6): 2 x WavLMNoLayerNormConvLayer(
        (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
        (activation): GELUActivation()
      )
    )
  )
  (feature_projection): WavLMFeatureProjection(
    (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (projection): Linear(in_features=512, out_features=768, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): WavLMEncoder(
    (pos_conv_embed): WavLMPositionalConvEmbedding(
      (conv): Parametrized

In [15]:
# Select first point of the dataset
print("Audio Shape:", dataset[0]["audio"]["array"].shape)
input_values = feature_extractor(dataset[0]["audio"]["array"], sampling_rate=sampling_rate, return_tensors="pt")['input_values']
print("Input Values Shape:", input_values.shape)

Audio Shape: (93680,)
Input Values Shape: torch.Size([1, 93680])


In [16]:
with torch.no_grad():
    outputs = model(input_values)

In [17]:
# Extract encoder output (this includes feature extraction)
encoder_output = outputs['last_hidden_state']  # Shape: (1, seq_len, feature_dim)
print("Encoder Output Shape:", encoder_output.shape)

Encoder Output Shape: torch.Size([1, 292, 768])


In [18]:
encoder_output

tensor([[[-0.1524, -0.2139, -0.1196,  ...,  1.2129,  0.2217, -0.3977],
         [-0.1470, -0.2864, -0.0996,  ...,  1.2637,  0.2217, -0.4654],
         [-0.1055, -0.3247, -0.1150,  ...,  1.3419,  0.2127, -0.4782],
         ...,
         [ 0.0136, -0.2798, -0.4029,  ...,  0.9122,  0.2058, -0.3439],
         [-0.0423, -0.2395, -0.4088,  ...,  0.9519,  0.1429, -0.4677],
         [-0.1248, -0.2294, -0.2764,  ...,  0.9044,  0.1780, -0.5477]]])

#### 1.3.2 Quantizer

In [19]:
# Apply K-Means clustering
num_clusters = 50
features = encoder_output.squeeze(0).numpy()
features.shape  # Shape: (seq_len, feature_dim)

(292, 768)

In [20]:
kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
kmeans.fit(features)

KMeans(n_clusters=50, n_init=10, random_state=42)

In [21]:
# Convert encoder output to discrete representations
quantized_ids = kmeans.predict(features)
print("Discrete Representation (First 20 IDs):", quantized_ids[:20])
print(quantized_ids.shape)

Discrete Representation (First 20 IDs): [12 12 12 12 12  6  6  6  6  6  6  6 29 29 29 29 29 29  3  3]
(292,)


## 2. Quantizer training & testing - all the basics

In this part you can find easy to use and general function in order to perform your test. Let me do some detail on what to find here :

- Useful function
  - `augment_audio` : basic augmentation function for some test (use the structure for new augmentation function)
  - `compute_ctc_loss` : loss function based on the one used in the paper
  - `MPLQuantizer` : class for quantizer training
  - `Quantizer0` : function to use the kmeans during computation
- Dataset preprocessing
  - `preprocess_and_save_features` : use a given model first part to preprocess a dataset of clean and perturbed features
  -`pad_batch` : pad embedded signals during batching
- Training
  - `train_quantizer` : training process used in the paper (*Attention:* if you use E1 to train E2, make sure to feed this function a callable i.e `E1 = lambda x: E1.predict(x)`)
- UED computation
  - `compute_ued` : compute the UED distance introduced in the paper
    - If you want to test E0 transform the function into a class (`E0_quantizer`)
- Some more basic function
  - `random_augment_audio` : if you want to train using a random augmentation you can use that (need to take as input a proper augmentation function)
  - `compute_ued_between_quantizer` : if you want to compute the Levensthein distance between E0 and E1 output for example
- ABX metric
  - `FullModel` : if you want to test a model on ABX, you need to give the encoder, quantizer, etc. this class facilitate this by reconstructing the model up to the quantizer.
  - `ABX_score` : compute the ABX score (*Attention:* the score rely on a distance, the first introduction of ABX used DTW metric, in the paper they are not clear on wether they use Levensthein for it. I did not implement DTW here but you can easily feed it to `ABX_score` using the parameter `distance`)


### 2.0 Usefull function

In [33]:
import types
import torch
import random
import torchaudio
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
import os
from Levenshtein import distance as levenshtein_distance

In [23]:
def augment_audio(audio: torch.Tensor, sr: int, augmentation_type: str, **kwargs) -> torch.Tensor:
    """
    Apply an augmentation.

    Parameters:
      audio (torch.Tensor): The input audio waveform (shape: [channels, samples]).
      sr (int): Sampling rate of the audio.
      augmentation_type (str): The type of augmentation to perform.
                                 Options: "gaussian_noise".
      **kwargs: Additional keyword arguments for specific augmentations.

    Returns:
      torch.Tensor: The augmented audio waveform.
    """
    # Gaussian noise :
    if augmentation_type == "gaussian_noise":
        noise_level = kwargs.get("noise_level", 0.005)
        noise = torch.randn_like(audio)
        augmented_audio = audio + noise_level * noise
        return augmented_audio

    else:
        raise ValueError(f"Unknown augmentation type: {augmentation_type}")

In [10]:
def compute_ctc_loss(
    features_: torch.Tensor,
    perturbed_features_: torch.Tensor,
    E0: callable,
    E1: nn.Module,
    blank_token: int = 0,
    target_lengths: torch.Tensor = None,
    input_lengths: torch.Tensor = None,
    target_threshold: float = 0.4
) -> torch.Tensor:
    """
    Compute the CTC loss as described in the paper :
      1) Compute E0(f(x)) to obtain targets tokens.
      2) Compute E1(f(g(x))) to obtain predictions logits.
      3) Compute CTC alignment between target tokens and predictions.

    Args:
      features_ (torch.Tensor): Input features (shape: [batch_size, time_steps, feat_dim]).
      perturbed_features_ (torch.Tensor): Augmented features (shape: [batch_size, time_steps, feat_dim]).
      E0 (nn.Module): Pretrained quantizer -> outputs discrete token IDs.
      E1 (nn.Module): Trainable quantizer (IN PAPER : MLP) -> outputs logits over discrete tokens.
      blank_token (int): Index for the CTC blank token.
      target_threshold (float): The percentage of the input size used for target (bigger than 0.5 would results in problems).

    Returns:
      torch.Tensor: Scalar CTC loss.
    """
    # 1) Get target tokens from original input
    assert isinstance(E0, types.FunctionType), "E0 must be a callable function, if you use a trained quantizer do : E0 = lambda x: E1.predict(x)"
    with torch.no_grad():
        target_tokens = E0(features_)  # [batch_size, target_len]

    # 2) Augment input, then get prediction logits
    prediction_logits = E1(perturbed_features_)    # [batch_size, time_steps, vocab_size]

    # Convert logits to log-probabilities for CTC
    prediction_log_probs = F.log_softmax(prediction_logits, dim=-1)  # [batch_size, time_steps, vocab_size]

    # Prepare lengths for CTC
    #    - input_lengths: how many time steps in the prediction output
    #    - target_lengths: how many tokens in the target
    batch_size, time_steps, vocab_size = prediction_log_probs.shape
    target_len = target_tokens.shape[1]

    if input_lengths is None:
        input_lengths = torch.full(
            size=(batch_size,),
            fill_value=time_steps,
            dtype=torch.long,
            device=prediction_log_probs.device
        )
    if target_lengths is None:
        target_lenghts = [int(input_lengths[i] * target_threshold) for i in range(batch_size)]
        target_lengths = torch.tensor(target_lenghts, dtype=torch.long, device=prediction_log_probs.device)

    target_tokens = [target_tokens[i, :target_lengths[i]] for i in range(batch_size)]
    target_tokens = torch.cat(target_tokens, dim=0)

    assert (input_lengths > target_lengths).all(), "CTC usually requires input lengths > target lengths!"

    # 4) CTC expects [time, batch, vocab] for the log probabilities
    prediction_log_probs = prediction_log_probs.permute(1, 0, 2).contiguous()

    # Flatten target tokens for CTC. They must be a 1D tensor concatenated for the batch,
    # but we also need to pass the correct target_lengths for each sample (see doc but target_lenghts size impact a lot the loss).
    target_tokens = target_tokens.view(-1)

    # 5) Instantiate and compute the CTC loss
    ctc_loss_fn = nn.CTCLoss(blank=blank_token, zero_infinity=True)
    loss = ctc_loss_fn(prediction_log_probs, target_tokens, input_lengths, target_lengths)
    return loss

In [11]:
class MLPQuantizer(nn.Module):
    """Trainable quantizer E1 that outputs logits over discrete tokens."""
    def __init__(self, input_dim: int, vocab_size: int, hidden_dim = None, activation_fn = None):
        super().__init__()
        self.input_dim = input_dim
        self.vocab_size = vocab_size
        if hidden_dim is not None:
          if activation_fn is None:
            activation_fn = nn.ReLU
          if len(hidden_dim) == 0:
            hidden_dim = [hidden_dim]
          self.layer = []
          for idx, hidden_dim_ in enumerate(hidden_dim):
            if idx == 0:
              self.layer.append(nn.Linear(input_dim, hidden_dim_))
              self.layer.append(activation_fn())
            else:
              self.layer.append(nn.Linear(hidden_dim[idx-1], hidden_dim_))
              self.layer.append(activation_fn())
          self.layer.append(nn.Linear(hidden_dim[-1], vocab_size))
          self.layer = nn.Sequential(*self.layer)

        else:
          self.layer = nn.Linear(input_dim, vocab_size)

    def forward(self, features):
        """
        Args:
            features: Tensor of shape [batch, time_step, feature]
        Returns:
            logits: Tensor of shape [batch, time_step, vocab_size]
        """
        logits = self.layer(features)
        return logits

    def predict(self, features):
        """
        Args:
            features: Tensor of shape [batch, time_step, feature]
        Returns:
            predictions: Tensor of shape [batch, time_step]
        """
        logits = self(features)
        predictions = logits.argmax(dim=-1)
        return predictions

In [74]:
def Quantizer0(features_: torch.Tensor, kmeans) -> torch.Tensor:
    """
    Discretize features using a pretrained KMeans model.

    Args:
        features_ (torch.Tensor): Input features (shape: [batch_size, time_steps, feat_dim]).
        kmeans (KMeans): Pretrained KMeans model.

    Returns:
        torch.Tensor: Discrete representations (shape: [batch_size, time_steps]).
    """
    batch_size = features_.shape[0]
    quantized_ids = torch.zeros(batch_size, features_.shape[1], dtype=torch.long)
    for idx in range(batch_size):
        _features_ = features_[idx].view(-1, features_.shape[-1]).cpu().detach().numpy()
        quantized_ids[idx] = torch.tensor(kmeans.predict(_features_))
    return quantized_ids

In [24]:
# test ctc loss
vocab_size = 50

# Load quantizer E1
E1 = MLPQuantizer(input_dim=768, vocab_size=vocab_size)
E1.train()

# Load perturbed features
perturbed_audio = augment_audio(torch.Tensor(dataset[0]["audio"]["array"]), sr=sampling_rate, augmentation_type="gaussian_noise", noise_level=0.5)
perturbed_features = feature_extractor(perturbed_audio, sampling_rate=sampling_rate, return_tensors="pt")['input_values']
perturbed_features = model(perturbed_features)['last_hidden_state']

# Load quantizer E0
kmeans = KMeans(n_clusters=vocab_size, random_state=42, n_init=10)
kmeans.fit(encoder_output.squeeze(0).numpy())
E0 = lambda x: Quantizer0(x, kmeans)

# compute ctc loss
features = encoder_output
print("Features Shape:", features.shape)

ctc_loss = compute_ctc_loss(features, perturbed_features, E0, E1)
print("CTC Loss:", ctc_loss.item())

E1.eval()
E1.predict(perturbed_features)



Features Shape: torch.Size([1, 292, 768])
CTC Loss: 8.093291282653809


tensor([[16, 41, 41, 41, 41, 41, 41, 22,  5, 22, 22, 22, 22, 22, 22, 22, 17, 17,
         17, 17, 22, 22, 22, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20,
         20, 20, 20, 20, 20,  2,  2,  2,  2,  2,  2,  2, 20,  2,  2, 20, 20, 20,
          2,  2,  2,  2,  2, 20,  2,  2,  2, 20,  2, 20, 20, 20, 20, 20, 20, 20,
          2, 20,  2,  2,  2,  2,  2,  2,  2, 20,  2,  2, 20, 20, 20,  2,  2,  2,
          2, 20, 36, 20, 20, 36,  2,  2,  2, 20, 20, 20,  2,  2,  2,  2,  2,  2,
          2,  2,  2, 20,  2,  2, 36,  2,  2,  2,  2, 20, 20,  2,  2, 20,  2,  2,
         20,  2, 20, 20, 20, 20, 36, 36, 36, 36, 36, 20, 36, 20, 20, 36, 36, 36,
         36, 36, 20, 36,  2, 20, 20, 20, 20, 20, 36, 20, 20, 36, 36, 36, 20,  2,
         20,  2,  2,  2, 20, 20, 36, 36, 36,  2,  2,  2,  2,  2,  2,  2,  2,  2,
         20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20,  2, 20, 20, 20, 20, 20, 20,
         20,  2, 20,  2,  2,  2,  2, 20,  2, 20, 20, 20, 20, 20, 20, 20, 20, 20,
         20, 20, 20, 20, 20,

### 2.1 Dataset preprocessing

In [25]:
from tqdm import tqdm

def preprocess_and_save_features(dataset, encoder, feature_extractor, augmentation_fn, sampling_rate, num_samples=100, save_path="precomputed_features.pt", hidden_state=None):

    N = min(num_samples, len(dataset))
    encoded_features = []
    perturbed_encoded_features = []

    for idx in tqdm(range(N), desc="Processing dataset"):
        audio = dataset[idx]["audio"]["array"]

        # Extract features
        features = feature_extractor(audio, sampling_rate=sampling_rate, return_tensors="pt")["input_values"]
        if hidden_state is not None:
          assert isinstance(hidden_state, int), "hidden_state must be an integer : Example, use 9th HuBERT hidden layer."
          features = encoder(features)["hidden_states"][hidden_state].squeeze(0)  # Remove batch dim
        else:
          features = encoder(features)["last_hidden_state"].squeeze(0)  # Remove batch dim

        encoded_features.append(features.cpu().detach().numpy())  # Convert to NumPy

        # Apply augmentation
        perturbed_audio = augmentation_fn(torch.Tensor(audio), sampling_rate)
        perturbed_features = feature_extractor(perturbed_audio, sampling_rate=sampling_rate, return_tensors="pt")["input_values"]
        if hidden_state is not None:
          perturbed_features = encoder(perturbed_features)["hidden_states"][hidden_state].squeeze(0)  # Remove batch dim
        else:
          perturbed_features = encoder(perturbed_features)["last_hidden_state"].squeeze(0)  # Remove batch dim

        perturbed_encoded_features.append(perturbed_features.cpu().detach().numpy())  # Convert to NumPy

    # Save as a NumPy file or Torch tensor
    torch.save(encoded_features, save_path)
    torch.save(perturbed_encoded_features, save_path.replace(".pt", "_perturbed.pt"))
    print(f"Precomputed features saved at {save_path}")

In [26]:
def pad_batch(batch):
    """
    Pads both the dataset and perturbed dataset in the batch to the same sequence length
    and returns original lengths, padded dataset, and perturbed dataset.
    """
    # Extract original sequence lengths for both datasets
    dataset_tensors, perturbed_tensors = zip(*batch)  # Unzip dataset and perturbed dataset

    # Get the sequence lengths
    lengths = [tensor.shape[0] for tensor in dataset_tensors]
    new_dataset = dataset_tensors + perturbed_tensors

    # Pad both datasets
    padded_new_dataset = pad_sequence(new_dataset, batch_first=True, padding_value=0.0)
    padded_dataset = padded_new_dataset[:len(dataset_tensors)]
    padded_perturbed = padded_new_dataset[len(dataset_tensors):]

    return lengths, padded_dataset, padded_perturbed

In [40]:
## You can always use this routine before anything to get the preprocessed datasets and your E0 super kmeans.
# load dataset
num_samples = 10
vocab_size = 50
augmentation_fn = lambda x, sampling_rate: augment_audio(x, sr=sampling_rate, augmentation_type="gaussian_noise", noise_level=0.01)
file_name = "precomputed_features.pt"
if not os.path.exists(file_name):
    preprocess_and_save_features(dataset, model, feature_extractor, augmentation_fn, sampling_rate, num_samples=num_samples, save_path=file_name)

# Load precomputed features
features = torch.load(file_name, weights_only=False)
perturbed_features = torch.load(file_name.replace(".pt", "_perturbed.pt"), weights_only=False)

features = [torch.tensor(f) if isinstance(f, np.ndarray) else f for f in features] # If numpy array then convert to torch tensor
perturbed_features = [torch.tensor(f) if isinstance(f, np.ndarray) else f for f in perturbed_features] # If numpy array then convert to torch tensor

# FIT a super kmeans on all data that will be used for E0
super_kmeans = KMeans(n_clusters=vocab_size, random_state=42, n_init=10)
fitting_features = torch.cat(features, dim=0)
super_kmeans.fit(fitting_features.numpy())

Processing dataset: 100%|██████████| 10/10 [01:32<00:00,  9.25s/it]


Precomputed features saved at precomputed_features.pt


KMeans(n_clusters=50, n_init=10, random_state=42)

### 2.2 Training

In [27]:
def train_quantizer(E0, E1, dataset, perturbed_dataset, num_epochs=10, batch_size=16, learning_rate=1e-3):
    """
    Train the quantizer E1 using the CTC loss while keeping E0 frozen.

    Args:
        E0: Pretrained quantizer.
        E1: Trainable quantizer.
        dataset: Dataset containing features.
        perturbed_dataset: Dataset containing perturbed features.
        num_epochs: Number of training epochs.
        batch_size: Batch size for training.
        learning_rate: Learning rate for optimizer.

    Returns:
        Trained quantizer E1.
    """

    # Freeze E0 (if it has parameters)
    if hasattr(E0, 'parameters') and any(p.requires_grad for p in E0.parameters()):
        for param in E0.parameters():
            param.requires_grad = False

    # Set E1 to training mode
    E1.train()

    # Define optimizer for E1
    optimizer = optim.Adam(E1.parameters(), lr=learning_rate)

    # Create DataLoader
    dataloader = DataLoader(
        list(zip(dataset, perturbed_dataset)),
        batch_size=batch_size,
        shuffle=True,
        collate_fn=pad_batch
    )

    for epoch in range(num_epochs):
        total_loss = 0.0
        E1.train()
        for lengths, batch, perturbed_batch in dataloader:
            optimizer.zero_grad()

            # Load clean features
            clean_features_ = batch

            # Load perturbed features
            perturbed_features_ = perturbed_batch

            # Compute CTC loss
            loss = compute_ctc_loss(clean_features_, perturbed_features_, E0, E1, input_lengths=torch.LongTensor(lengths), target_threshold=0.35)
            # Backpropagation
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(dataloader)
        print(f"Epoch {epoch + 1}/{num_epochs} - CTC Loss: {avg_loss:.4f}")

    return E1

In [42]:
print(len(features), len(perturbed_features))

10 10


In [75]:
# train quantizer
vocab_size = super_kmeans.n_clusters
# Load quantizer E0
E0 = lambda x: Quantizer0(x, super_kmeans)

# Load quantizer E1 (adapt parameters if needed)
E1 = MLPQuantizer(input_dim=768, vocab_size=vocab_size)

E1 = train_quantizer(E0, E1, features, perturbed_features, num_epochs=15, batch_size=5, learning_rate=1e-3)
# save model E1
torch.save(E1.state_dict(), "E1.pth")

Epoch 1/15 - CTC Loss: 8.4529
Epoch 2/15 - CTC Loss: 6.8845
Epoch 3/15 - CTC Loss: 5.5332
Epoch 4/15 - CTC Loss: 4.5470
Epoch 5/15 - CTC Loss: 3.9982
Epoch 6/15 - CTC Loss: 3.7531
Epoch 7/15 - CTC Loss: 3.6330
Epoch 8/15 - CTC Loss: 3.5590
Epoch 9/15 - CTC Loss: 3.4957
Epoch 10/15 - CTC Loss: 3.4423
Epoch 11/15 - CTC Loss: 3.3931
Epoch 12/15 - CTC Loss: 3.3468
Epoch 13/15 - CTC Loss: 3.3038
Epoch 14/15 - CTC Loss: 3.2647
Epoch 15/15 - CTC Loss: 3.2250


In [44]:
# train a new quantizer on past quantizer
E1 = MLPQuantizer(input_dim=768, vocab_size=50)
E1.load_state_dict(torch.load("E1.pth"))
vocab_size = E1.vocab_size
# Load quantizer E1
E1_fn = lambda x: E1.predict(x)

# Load quantizer E2
E2 = MLPQuantizer(input_dim=768, vocab_size=vocab_size)

E2 = train_quantizer(E1_fn, E2, features, perturbed_features, num_epochs=15, batch_size=5, learning_rate=1e-3)
# save model E2
torch.save(E2.state_dict(), "E2.pth")
E2.predict(torch.Tensor(perturbed_features[0]))

Epoch 1/15 - CTC Loss: 9.2689
Epoch 2/15 - CTC Loss: 8.5793
Epoch 3/15 - CTC Loss: 7.9004
Epoch 4/15 - CTC Loss: 7.2312
Epoch 5/15 - CTC Loss: 6.5762
Epoch 6/15 - CTC Loss: 5.9298
Epoch 7/15 - CTC Loss: 5.3173
Epoch 8/15 - CTC Loss: 4.7400
Epoch 9/15 - CTC Loss: 4.1885
Epoch 10/15 - CTC Loss: 3.6716
Epoch 11/15 - CTC Loss: 3.1948
Epoch 12/15 - CTC Loss: 2.7450
Epoch 13/15 - CTC Loss: 2.3315
Epoch 14/15 - CTC Loss: 1.9489
Epoch 15/15 - CTC Loss: 1.5966


tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0])

### 2.3 UED computation

In [45]:
# load E1
E1 = MLPQuantizer(input_dim=768, vocab_size=50)
E1.load_state_dict(torch.load("E1.pth"))

<All keys matched successfully>

In [28]:
def deduplicate(sequence):
    """Removes consecutive sames tokens of a tokens list."""
    deduplicated_seq = []
    prev_token = -1
    for token in sequence:
        if token != prev_token:
            deduplicated_seq.append(token)
        prev_token = token
    return deduplicated_seq

def compute_ued(dataset, perturbed_dataset, quantizer, deduplication=True):
    """
    Compute the Unit Edit Distance (UED) metric.

    Args:
        dataset: Dataset containing clean features.
        perturbed_dataset: Dataset containing perturbed features.
        quantizer: Quantizer model.

    Returns:
        The average normalized Levenshtein distance across the dataset.
    """
    total_distance = 0
    total_frames = 0

    for (x, augmented_x) in zip(dataset, perturbed_dataset):
        x = x.unsqueeze(0)
        augmented_x = augmented_x.unsqueeze(0)
        quantized_x = quantizer.predict(x).flatten()  # Convert to token sequence
        quantized_aug_x = quantizer.predict(augmented_x).flatten()
        # deduplication : supress redondant tokens
        if deduplication:
            quantized_x = deduplicate(quantized_x.tolist())
            quantized_aug_x = deduplicate(quantized_aug_x.tolist())
        else:
            quantized_x = quantized_x.tolist()
            quantized_aug_x = quantized_aug_x.tolist()
        # Compute Levenshtein distance
        lev_dist = levenshtein_distance(quantized_x, quantized_aug_x)
        total_distance += lev_dist / len(quantized_x)  # Divide by T'_x
    return total_distance

In [29]:
class E0_quantizer():
    def __init__(self, kmeans_quantizer_fn):
        self.quantizer = kmeans_quantizer_fn

    def predict(self, x):
        return self.quantizer(x)

In [48]:
# compute ued E0:
E0_fn = lambda x: Quantizer0(x, super_kmeans)
E0 = E0_quantizer(E0_fn)
ued = compute_ued(features, perturbed_features, E0)
print("UED:", ued)

UED: 2.719143312028647


In [49]:
# compute ued E1:
ued = compute_ued(features, perturbed_features, E1)
print("UED:", ued)

UED: 4.595604395604395


In [50]:
# Make test dataset
num_samples = 10
test_dataset = [dataset[num_samples -1 + idx] for idx in range(num_samples)]
augmentation_fn = lambda x, sampling_rate: augment_audio(x, sr=sampling_rate, augmentation_type="gaussian_noise", noise_level=0.01)
file_name = "test_precomputed_features.pt"
if not os.path.exists(file_name):
    preprocess_and_save_features(dataset, model, feature_extractor, augmentation_fn, sampling_rate, num_samples=num_samples, save_path=file_name)

# Load precomputed features
test_features = torch.load(file_name, weights_only=False)
test_perturbed_features = torch.load(file_name.replace(".pt", "_perturbed.pt"), weights_only=False)

test_features = [torch.tensor(f) if isinstance(f, np.ndarray) else f for f in test_features] # If numpy array then convert to torch tensor
test_perturbed_features = [torch.tensor(f) if isinstance(f, np.ndarray) else f for f in test_perturbed_features] # If numpy array then convert to torch tensor

Processing dataset: 100%|██████████| 10/10 [01:27<00:00,  8.71s/it]


Precomputed features saved at test_precomputed_features.pt


In [51]:
# compute ued E0:
E0_fn = lambda x: Quantizer0(x, super_kmeans)
E0 = E0_quantizer(E0_fn)
ued = compute_ued(test_features, test_perturbed_features, E0)
print("UED:", ued)

# compute ued E1:
ued = compute_ued(test_features, test_perturbed_features, E1)
print("UED:", ued)

UED: 2.8592443035728214
UED: 4.32026862026862


### 2.4 Some more basic functions

In [30]:
import random

def random_augment_audio(audio: torch.Tensor, sr: int, augmenter_audio: callable, augmentation_type_list: list, **kwargs) -> torch.Tensor:
    """
    Apply a random augmentation.

    augmenter_audio : your augmenter function (see mine in 2.0 for reference)
    augmentation_type : A list of augmentation functions to apply.
    **kwargs: Additional keyword arguments for specific augmentations

    /!\ remember that when you define the augmentation_fn for training use EXAMPLE :
    augmentation_type_list = ["gaussian_noise", ...]
    augmentation_fn = lambda x, sr: random_augment_audio(x, sr, augmentation_type_list, **kwargs)
    """
    # pick random augmentation
    augmentation_type = random.choice(augmentation_type_list)
    return augmenter_audio(audio, sr, augmentation_type, **kwargs)


In [31]:
def compute_ued_between_quantizer(dataset, perturbed_dataset, tested_quantizer, target_quantizer):
    """
    Compute the Unit Edit Distance (UED) metric.

    Args:
        dataset: Dataset containing clean features.
        perturbed_dataset: Dataset containing perturbed features.
        tested_quantizer: Quantizer model to be tested on perturbed data.
        target_quantizer: Quantizer model to use for the test (be sure it has been trained on same data if not E0).

    Returns:
        The average normalized Levenshtein distance across the dataset.
    """
    total_distance = 0
    total_frames = 0

    for (x, augmented_x) in zip(dataset, perturbed_dataset):
        x = x.unsqueeze(0)
        augmented_x = augmented_x.unsqueeze(0)
        quantized_x = target_quantizer.predict(x).flatten()  # Convert to token sequence
        quantized_aug_x = tested_quantizer.predict(augmented_x).flatten()
        # Compute Levenshtein distance
        lev_dist = levenshtein_distance(quantized_x.tolist(), quantized_aug_x.tolist())
        total_distance += lev_dist / len(quantized_x)  # Divide by T'_x

    return total_distance

In [54]:
# compute ued E1 versus E0:
vocab_size = 50
E0_fn = lambda x: Quantizer0(x, super_kmeans)
E0 = E0_quantizer(E0_fn)
E1 = MLPQuantizer(input_dim=768, vocab_size=50)
E1.load_state_dict(torch.load("E1.pth"))
ued = compute_ued_between_quantizer(features, perturbed_features, E1, E0)
print("UED:", ued)

UED: 9.828692935855717


### 2.5 ABX metric

In [68]:
class FullModel(nn.Module):
    def __init__(self, features_extractor, model, quantizer, sampling_rate, hidden_state=None):
        super().__init__()
        self.features_extractor = features_extractor
        self.model = model
        self.quantizer = quantizer
        self.sampling_rate = sampling_rate
        self.hidden_state = hidden_state

    def forward(self, x):
        features = self.features_extractor(x, sampling_rate=self.sampling_rate, return_tensors="pt")['input_values']
        if self.hidden_state is not None :
          features = self.model(features)['hidden_states'][self.hidden_state]
        else :
          features = self.model(features)['last_hidden_state']
        token = self.quantizer.predict(features.unsqueeze(0)).flatten()
        return token

In [56]:
# ------------------------- load our model

E1 = MLPQuantizer(input_dim=768, vocab_size=50)
E1.load_state_dict(torch.load("E1.pth"))

full_model = FullModel(feature_extractor, model, E1, sampling_rate)

# ------------------------- I advise you to stream the dataset.
dataset = load_dataset('gilkeyio/librispeech-alignments', split='dev_clean', streaming=True)
for sample in dataset.take(1):
    if "audio" in sample:
        print("Sampling rate:", sample["audio"]["sampling_rate"])
        sampling_rate = sample["audio"]["sampling_rate"]
    else:
        print("No audio field in this sample.")
# ------------------------- Build a phoneme dictionary.
phoneme_dict = {}
num_examples = 100
counter = 0
for entry in dataset:
    phonemes = entry['phonemes']
    for ph in phonemes:
        phoneme = ph["phoneme"]
        if phoneme not in phoneme_dict:
            phoneme_dict[phoneme] = []
        end_time = ph["end"]
        start_time = ph["start"]
        # time are of the form 0.21, 0.22, etc.
        audio = entry["audio"]["array"][int(start_time * sampling_rate):int(end_time * sampling_rate)]
        phoneme_dict[phoneme].append(audio)
    counter += 1
    if counter >= num_examples:
        break

# ------------------------- Create ABX triplets.
# For each triplet:
#  - A and X come from the same phoneme class.
#  - B comes from a different phoneme class.
def create_abx_triplets(phoneme_dict, num_triplets=100):
    triplets = []
    phoneme_classes = list(phoneme_dict.keys())
    for _ in range(num_triplets):
        # Choose a phoneme class that has at least 2 of these for A and X.
        valid_classes = [ph for ph in phoneme_classes if len(phoneme_dict[ph]) >= 2]
        if not valid_classes:
            break
        class_A = random.choice(valid_classes)
        A_X = random.sample(phoneme_dict[class_A], 2)
        A_token, X_token = A_X[0], A_X[1]
        # Choose a different class for B (needs at least one of these)
        valid_B_classes = [ph for ph in phoneme_classes if ph != class_A and len(phoneme_dict[ph]) >= 1]
        if not valid_B_classes:
            break
        class_B = random.choice(valid_B_classes)
        B_token = random.choice(phoneme_dict[class_B])
        triplets.append((A_token, B_token, X_token))
    return triplets

triplets = create_abx_triplets(phoneme_dict, num_triplets=100)
print(f"Created {len(triplets)} ABX triplets.")

# ------------------------- Define a distance function. Here we use Levenhstein.
def distance_of_lev(output, target):
    return levenshtein_distance(output.tolist(), target.tolist())

# ------------------------- Compute ABX scores.
def compute_abx_score(triplets, model, distance = distance_of_lev):
    errors = 0
    total = len(triplets)
    for A_ph, B_ph, X_ph in triplets:
        A_token = model(A_ph)
        B_token = model(B_ph)
        X_token = model(X_ph)
        d_AX = distance(A_token, X_token)
        d_BX = distance(B_token, X_token)
        # For ABX, if X is closer to A than to B, it's correct.
        if d_AX > d_BX:
            errors += 1
    return errors / total

# -------------------------

abx_error_rate = compute_abx_score(triplets, full_model)
print("ABX error rate:", abx_error_rate)


README.md:   0%|          | 0.00/6.34k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/49 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/66 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/49 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/66 [00:00<?, ?it/s]

Sampling rate: 16000
Created 100 ABX triplets.
ABX error rate: 0.3


In [69]:
def distance_of_lev(output, target):
        return levenshtein_distance(output.tolist(), target.tolist())

def ABX_score(full_model, _distance_ = distance_of_lev, num_examples=100, num_triplets=100):
    # ------------------------- I advise you to stream the dataset.
    dataset = load_dataset('gilkeyio/librispeech-alignments', split='dev_clean', streaming=True)
    for sample in dataset.take(1):
        if "audio" in sample:
            print("Sampling rate:", sample["audio"]["sampling_rate"])
            sampling_rate = sample["audio"]["sampling_rate"]
        else:
            print("No audio field in this sample.")
    # ------------------------- Build a phoneme dictionary.
    phoneme_dict = {}
    counter = 0
    for entry in dataset:
        phonemes = entry['phonemes']
        for ph in phonemes:
            phoneme = ph["phoneme"]
            if phoneme not in phoneme_dict:
                phoneme_dict[phoneme] = []
            end_time = ph["end"]
            start_time = ph["start"]
            # time are of the form 0.21, 0.22, etc.
            audio = entry["audio"]["array"][int(start_time * sampling_rate):int(end_time * sampling_rate)]
            phoneme_dict[phoneme].append(audio)
        counter += 1
        if counter >= num_examples:
            break

    # ------------------------- Create ABX triplets.
    # For each triplet:
    #  - A and X come from the same phoneme class.
    #  - B comes from a different phoneme class.
    def create_abx_triplets(phoneme_dict, num_triplets=num_triplets):
        triplets = []
        phoneme_classes = list(phoneme_dict.keys())
        for _ in range(num_triplets):
            # Choose a phoneme class that has at least 2 of these for A and X.
            valid_classes = [ph for ph in phoneme_classes if len(phoneme_dict[ph]) >= 2]
            if not valid_classes:
                break
            class_A = random.choice(valid_classes)
            A_X = random.sample(phoneme_dict[class_A], 2)
            A_token, X_token = A_X[0], A_X[1]
            # Choose a different class for B (needs at least one of these)
            valid_B_classes = [ph for ph in phoneme_classes if ph != class_A and len(phoneme_dict[ph]) >= 1]
            if not valid_B_classes:
                break
            class_B = random.choice(valid_B_classes)
            B_token = random.choice(phoneme_dict[class_B])
            triplets.append((A_token, B_token, X_token))
        return triplets

    triplets = create_abx_triplets(phoneme_dict, num_triplets=num_triplets)
    print(f"Created {len(triplets)} ABX triplets.")

    # ------------------------- Compute ABX scores.
    def compute_abx_score(triplets, model, distance = _distance_):
        errors = 0
        total = len(triplets)
        for A_ph, B_ph, X_ph in triplets:
            A_token = model(A_ph)
            B_token = model(B_ph)
            X_token = model(X_ph)
            d_AX = distance(A_token, X_token)
            d_BX = distance(B_token, X_token)
            # For ABX, if X is closer to A than to B, it's correct.
            if d_AX > d_BX:
                errors += 1
        return errors / total

    # -------------------------

    abx_error_rate = compute_abx_score(triplets, full_model)
    print("ABX error rate:", abx_error_rate)

In [58]:
# ------------------------- load our model

E1 = MLPQuantizer(input_dim=768, vocab_size=50)
E1.load_state_dict(torch.load("E1.pth"))

full_model = FullModel(feature_extractor, model, E1, sampling_rate)

ABX_score(full_model, num_examples=100, num_triplets=100)

Resolving data files:   0%|          | 0/49 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/66 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/49 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/66 [00:00<?, ?it/s]

Sampling rate: 16000
Created 100 ABX triplets.
ABX error rate: 0.32


## 3. Augmentation

### 3.0 Dataset

In [59]:
dataset = load_dataset("hf-internal-testing/librispeech_asr_demo", "clean", split="validation", trust_remote_code=True)
dataset = dataset.sort("id")
sampling_rate = dataset.features["audio"].sampling_rate

### 3.1 Useful functions for auditive tests

In [60]:
import matplotlib.pyplot as plt
from IPython.display import Audio, display

def play_audio(audio, sr=16000, title="audio"):
    audio_numpy = audio.squeeze().numpy()
    print(f"Listen : {title}")
    display(Audio(audio_numpy, rate=sr))

def visualize_and_play(audio, augmented_audio, sr=16000):
    play_audio(audio, sr, "original audio")
    play_audio(augmented_audio, sr, "augmented audio")

### 3.2 Audio effects (basic definitions these effects with simple tests to tune params)

There are 11 augmentations : gaussian noise, time stretch (or changing pitch, since the time stretch does not keep the original frequencies), clipping, filters (lowpass, highpass, bandpass), adding bips (big and little ones), adding echo, missing samples and changing volume.

#### 3.2.1 Audio effect of adding noise

In [61]:
# check gaussian noise
audio=torch.tensor([1,2,3,4,5,6,7,8,9], dtype=torch.float)

print(" Audio before transform : ")
print(audio)

noise=torch.randn_like(audio)

audio_with_noise=audio+noise

print(" Audio after adding noise : ")
print(audio_with_noise)

print("Audio after multiplying everything by 0.001 : ")
print(0.01*audio)

 Audio before transform : 
tensor([1., 2., 3., 4., 5., 6., 7., 8., 9.])
 Audio after adding noise : 
tensor([0.2375, 1.6952, 3.4662, 4.8031, 4.0116, 4.4395, 7.7455, 7.7270, 8.3154])
Audio after multiplying everything by 0.001 : 
tensor([0.0100, 0.0200, 0.0300, 0.0400, 0.0500, 0.0600, 0.0700, 0.0800, 0.0900])


In [ ]:
audio=torch.tensor(dataset[0]["audio"]["array"])
print(f"shape of the 'audio' signal: {audio.shape}")
noise = torch.randn_like(audio)
noisy = audio+0.01*noise # put >=0.01
visualize_and_play(audio, noisy, sr=16000)

#### 3.2.2 Audio effect of changing speed

In [63]:
# check gaussian noise
audio=torch.tensor([1,2,3,4,5,6,7,8,9], dtype=torch.float)

print(" Audio before transform : ")
print(audio)

# change speed
def speed_change(audio, rate):
    indices = torch.arange(0, audio.shape[0], rate)
    indices = indices.long()
    return audio[indices]

faster_audio = speed_change(audio, 0.5)
print(" Audio after speed change : ")
print(faster_audio)

 Audio before transform : 
tensor([1., 2., 3., 4., 5., 6., 7., 8., 9.])
 Audio after speed change : 
tensor([1., 1., 2., 2., 3., 3., 4., 4., 5., 5., 6., 6., 7., 7., 8., 8., 9., 9.])


In [ ]:
audio=torch.tensor(dataset[1]["audio"]["array"])
print(f"shape of the 'audio' signal: {audio.shape}")
audio_speed_change=speed_change(audio, 1.2)
visualize_and_play(audio, audio_speed_change, sr=16000)

#### 3.2.3 Audio effect of changing "pitch"

In [65]:
audio=torch.tensor([1,2,3,4,5,6,7,8,9], dtype=torch.float)
print(" Audio before transform : ")
print(audio)

# change pitch high
def pitch_shift(audio, n_steps):
    factor = 2**(n_steps/12)
    orig_len = audio.shape[0]
    new_len = int(orig_len/factor)
    indices = torch.arange(0, orig_len, step=factor).long()
    indices = indices[:min(new_len, len(indices))]
    return audio[indices]

n_steps=2
pitch_shifted_audio = pitch_shift(audio, n_steps)
print("Audio with modified pitch:")
print(pitch_shifted_audio)

 Audio before transform : 
tensor([1., 2., 3., 4., 5., 6., 7., 8., 9.])
Audio with modified pitch:
tensor([1., 2., 3., 4., 5., 6., 7., 8.])


In [ ]:
audio=torch.tensor(dataset[3]["audio"]["array"])
print(f"shape of the 'audio' signal: {audio.shape}")
audio_pitch_change=pitch_shift(audio, 10)
visualize_and_play(audio, audio_pitch_change, sr=16000)

#### 3.2.4 Audio effect of clipping

In [67]:
audio=torch.tensor([1,2,3,4,5,6,7,8,9], dtype=torch.float)
print(" Audio before transform : ")
print(audio)

clip_factor = 0.8
max_val = audio.abs().max()
threshold = clip_factor*max_val
augmented_audio = torch.clamp(audio, -threshold, threshold)
print("audio after clipping :")
print(augmented_audio)

 Audio before transform : 
tensor([1., 2., 3., 4., 5., 6., 7., 8., 9.])
audio after clipping :
tensor([1.0000, 2.0000, 3.0000, 4.0000, 5.0000, 6.0000, 7.0000, 7.2000, 7.2000])


In [ ]:
audio=torch.tensor(dataset[4]["audio"]["array"])
print(f"shape of the 'audio' signal: {audio.shape}")
threshold = 0.01*audio.abs().max() # <0.1
clipped_audio = torch.clamp(audio,-threshold,threshold)
visualize_and_play(audio, clipped_audio, sr=16000)

#### 3.2.5 Audio effect of filter (lowpass)

In [ ]:
audio=torch.tensor(dataset[5]["audio"]["array"])
print(f"shape of the 'audio' signal: {audio.shape}")
audio_np = audio.numpy()
fft_signal = np.fft.rfft(audio_np)
freqs = np.fft.rfftfreq(len(audio_np), 1/16000)
mask = freqs<=1000
filtered_fft = fft_signal*mask
filtered_signal = np.fft.irfft(filtered_fft)
audio_lowpass=torch.tensor(filtered_signal, dtype=audio.dtype)
visualize_and_play(audio, audio_lowpass, sr=16000)

#### 3.2.6 Audio effect of filter (bandpass)

In [ ]:
audio=torch.tensor(dataset[6]["audio"]["array"])
print(f"shape of the 'audio' signal: {audio.shape}")
audio_np = audio.numpy()
fft_signal = np.fft.rfft(audio_np)
freqs = np.fft.rfftfreq(len(audio_np), 1/16000)
mask = (freqs >= 500) & (freqs <= 1500)
filtered_fft = fft_signal*mask
filtered_signal = np.fft.irfft(filtered_fft)
audio_bandpass=torch.tensor(filtered_signal, dtype=audio.dtype)
visualize_and_play(audio, audio_bandpass, sr=16000)

#### 3.2.7 Audio effect of filter (highpass)

In [ ]:
audio=torch.tensor(dataset[7]["audio"]["array"])
print(f"shape of the 'audio' signal: {audio.shape}")
audio_np = audio.numpy()
fft_signal = np.fft.rfft(audio_np)
freqs = np.fft.rfftfreq(len(audio_np), 1/16000)
mask = freqs>=500
filtered_fft = fft_signal*mask
filtered_signal = np.fft.irfft(filtered_fft)
audio_highpass=torch.tensor(filtered_signal, dtype=audio.dtype)
visualize_and_play(audio, audio_highpass, sr=16000)

#### 3.2.8 Audio effects of adding many little bips

In [72]:
def bip(audio):
  audio_bipped=audio.clone()
  max=audio.abs().max()
  for k in range(audio.shape[0]):
    if k%2==0:
      audio_bipped[k]=max
  return audio_bipped

In [ ]:
audio=torch.tensor(dataset[8]["audio"]["array"])
print(f"shape of the 'audio' signal: {audio.shape}")
audio_bip= bip(audio)
visualize_and_play(audio, audio_bip, sr=16000)

#### 3.2.9 Audio effects of adding some big bips

In [74]:
def big_bip(audio):
  audio_bipped=audio.clone()
  max=audio.abs().max()
  for k in range(audio.shape[0]):
    if k%10000==0 and k+100<=audio.shape[0]:
      audio_bipped[k:k+100]=torch.tensor([max]*100)
  return audio_bipped

In [ ]:
audio=torch.tensor(dataset[9]["audio"]["array"])
print(f"shape of the 'audio' signal: {audio.shape}")
audio_big_bip= big_bip(audio)
visualize_and_play(audio, audio_big_bip, sr=16000)

#### 3.2.10 Audio effect of adding an echo

In [76]:
def echo(audio, nb_echos=5):
    nb_samples=audio.shape[0]
    '''
    plt.plot(audio)
    plt.title("audio")
    plt.show()
    '''
    echo_audio=audio.clone()
    for i in range(nb_echos):
      '''
      plt.plot((1/(i+1)**2)*audio[:(nb_samples-(1000*i))])
      plt.title(f"echo {i}")
      plt.show()
      '''
      echo_audio[(4000*i):]+=(1/(i+1)**2)*audio[:(nb_samples-(4000*i))]
    return echo_audio

audio=torch.tensor(dataset[10]["audio"]["array"])
print(f"shape of the 'audio' signal: {audio.shape}")
echo_audio=echo(audio, nb_echos=10)
visualize_and_play(audio, echo_audio, sr=16000)

shape of the 'audio' signal: torch.Size([89600])
Listen : original audio


Listen : augmented audio


### 3.3 Function augment_audio

In [77]:
def augment_audio(audio: torch.Tensor, sr: int, augmentation_type: str, **kwargs) -> torch.Tensor:
    """
    Apply an augmentation.

    Parameters:
      audio (torch.Tensor): The input audio waveform (shape: [channels, samples]).
      sr (int): Sampling rate of the audio.
      augmentation_type (str): The type of augmentation to perform.
                                 Options: "gaussian_noise", "time_stretch",
                                          "pitch_shift", "clipping", "lowpass", "bandpass", "highpass", "little_bips",
                                          "big_bips", "echo"
      **kwargs: Additional keyword arguments for specific augmentations.

    Returns:
      torch.Tensor: The augmented audio waveform.
    """
    # Gaussian noise :
    if augmentation_type == "gaussian_noise":
        noise_level = kwargs.get("noise_level", 0.005)
        noise = torch.randn_like(audio)
        augmented_audio = audio + noise_level * noise
        return augmented_audio

    # Audio effect of changing speed
    elif augmentation_type == "time_stretch":
        rate = kwargs.get("rate", 1.2)  # >1 speeds up, <1 slows down.
        return speed_change(audio, rate)

    # Audio effect of changing "pitch"
    elif augmentation_type == "pitch_shift":
        n_steps = kwargs.get("n_steps", 2)
        return pitch_shift(audio, n_steps)

    # Audio effect of clipping
    elif augmentation_type == "clipping":
        clip_factor = kwargs.get("clip_factor", 0.8)
        threshold = 0.01*audio.abs().max() # <0.1
        return torch.clamp(audio,-threshold,threshold)

    # Audio effect of filter (lowpass)
    elif augmentation_type == "lowpass":
      freqlim = kwargs.get("freqlim", 1000)
      audio_np = audio.numpy()
      fft_signal = np.fft.rfft(audio_np)
      freqs = np.fft.rfftfreq(len(audio_np), 1/16000)
      mask = freqs<=freqlim
      filtered_fft = fft_signal*mask
      filtered_signal = np.fft.irfft(filtered_fft)
      audio_lowpass=torch.tensor(filtered_signal, dtype=audio.dtype)
      return audio_lowpass

    # Audio effect of filter (bandpass)
    elif augmentation_type == "bandpass":
      freqlim1 = kwargs.get("freqlim1", 500)
      freqlim2 = kwargs.get("freqlim2", 1500)
      audio_np = audio.numpy()
      fft_signal = np.fft.rfft(audio_np)
      freqs = np.fft.rfftfreq(len(audio_np), 1/16000)
      mask = (freqs >= freqlim1) & (freqs <= freqlim2)
      filtered_fft = fft_signal*mask
      filtered_signal = np.fft.irfft(filtered_fft)
      audio_bandpass=torch.tensor(filtered_signal, dtype=audio.dtype)
      return audio_bandpass

    # Audio effect of filter (highpass)
    elif augmentation_type == "highpass":
      freqlim = kwargs.get("freqlim", 500)
      audio_np = audio.numpy()
      fft_signal = np.fft.rfft(audio_np)
      freqs = np.fft.rfftfreq(len(audio_np), 1/16000)
      mask = freqs>=freqlim
      filtered_fft = fft_signal*mask
      filtered_signal = np.fft.irfft(filtered_fft)
      audio_highpass=torch.tensor(filtered_signal, dtype=audio.dtype)
      return audio_highpass

    # Audio effects of adding many little bips
    elif augmentation_type == "little_bips":
      audio_bip=bip(audio)
      return audio_bip


    # Audio effects of adding some big bips
    elif augmentation_type == "big_bips":
      audio_big_bip= big_bip(audio)
      return audio_big_bip


    # Audio effect of adding an echo
    elif augmentation_type == "echo":
      nb_echos = kwargs.get("nb_echos", 5)
      echo_audio=echo(audio, nb_echos)
      return echo_audio

    else:
        raise ValueError(f"Unknown augmentation type: {augmentation_type}")

## 4. Quantizers outputs for UED distance computation

### 4.0 Some more augmentation

#### 4.0.1 Audio effect of corrupted samples (or missing samples)

In [ ]:
def corruption(audio, drop_rate=0.01):
    nb_samples=audio.shape[0]
    corrupted_audio=audio.clone()
    for i in range(nb_samples):
        if random.random()<drop_rate:
            corrupted_audio[i]=0
    return corrupted_audio

audio=torch.tensor(dataset[11]["audio"]["array"])
print(f"shape of the 'audio' signal: {audio.shape}")
corrupted_audio=corruption(audio, drop_rate=0.01)
visualize_and_play(audio, corrupted_audio, sr=16000)

#### 4.0.2 Audio effect of changing volume

In [ ]:
def volume_change(audio, factor=0.5):
    augmented_audio = audio * factor
    return augmented_audio

audio=torch.tensor(dataset[11]["audio"]["array"])
print(f"shape of the 'audio' signal: {audio.shape}")
changed_volume_audio=volume_change(audio)
# use of another function than visualize_and_play because of the normalization
print('Listen, audio')
display(Audio(audio, rate=16000, normalize=False))
print('Listen, augmented audio')
display(Audio(changed_volume_audio, rate=16000, normalize=False))

### 4.1 General use augment audio

In [35]:
def augment_audio(audio: torch.Tensor, sr: int, augmentation_type: str, **kwargs) -> torch.Tensor:
    """
    Apply an augmentation.

    Parameters:
      audio (torch.Tensor): The input audio waveform (shape: [channels, samples]).
      sr (int): Sampling rate of the audio.
      augmentation_type (str): The type of augmentation to perform.
                                 Options: "gaussian_noise", "time_stretch",
                                          "pitch_shift", "clipping", "lowpass", "bandpass", "highpass", "little_bips",
                                          "big_bips", "echo"
      **kwargs: Additional keyword arguments for specific augmentations.

    Returns:
      torch.Tensor: The augmented audio waveform.
    """
    # Gaussian noise :
    if augmentation_type == "gaussian_noise":
        noise_level = kwargs.get("noise_level", 0.005)
        noise = torch.randn_like(audio)
        augmented_audio = audio + noise_level * noise
        return augmented_audio

    # Audio effect of changing speed
    elif augmentation_type == "time_stretch":
        rate = kwargs.get("rate", 1.2)  # >1 speeds up, <1 slows down.
        return speed_change(audio, rate)

    # Audio effect of changing "pitch"
    elif augmentation_type == "pitch_shift":
        n_steps = kwargs.get("n_steps", 2)
        return pitch_shift(audio, n_steps)

    # Audio effect of clipping
    elif augmentation_type == "clipping":
        clip_factor = kwargs.get("clip_factor", 0.8)
        threshold = 0.01*audio.abs().max() # <0.1
        return torch.clamp(audio,-threshold,threshold)

    # Audio effect of filter (lowpass)
    elif augmentation_type == "lowpass":
      freqlim = kwargs.get("freqlim", 1000)
      audio_np = audio.numpy()
      fft_signal = np.fft.rfft(audio_np)
      freqs = np.fft.rfftfreq(len(audio_np), 1/16000)
      mask = freqs<=freqlim
      filtered_fft = fft_signal*mask
      filtered_signal = np.fft.irfft(filtered_fft)
      audio_lowpass=torch.tensor(filtered_signal, dtype=audio.dtype)
      return audio_lowpass

    # Audio effect of filter (bandpass)
    elif augmentation_type == "bandpass":
      freqlim1 = kwargs.get("freqlim1", 500)
      freqlim2 = kwargs.get("freqlim2", 1500)
      audio_np = audio.numpy()
      fft_signal = np.fft.rfft(audio_np)
      freqs = np.fft.rfftfreq(len(audio_np), 1/16000)
      mask = (freqs >= freqlim1) & (freqs <= freqlim2)
      filtered_fft = fft_signal*mask
      filtered_signal = np.fft.irfft(filtered_fft)
      audio_bandpass=torch.tensor(filtered_signal, dtype=audio.dtype)
      return audio_bandpass

    # Audio effect of filter (highpass)
    elif augmentation_type == "highpass":
      freqlim = kwargs.get("freqlim", 500)
      audio_np = audio.numpy()
      fft_signal = np.fft.rfft(audio_np)
      freqs = np.fft.rfftfreq(len(audio_np), 1/16000)
      mask = freqs>=freqlim
      filtered_fft = fft_signal*mask
      filtered_signal = np.fft.irfft(filtered_fft)
      audio_highpass=torch.tensor(filtered_signal, dtype=audio.dtype)
      return audio_highpass

    # Audio effects of adding many little bips
    elif augmentation_type == "little_bips":
      audio_bip=bip(audio)
      return audio_bip


    # Audio effects of adding some big bips
    elif augmentation_type == "big_bips":
      audio_big_bip= big_bip(audio)
      return audio_big_bip


    # Audio effect of adding an echo
    elif augmentation_type == "echo":
      nb_echos = kwargs.get("nb_echos", 5)
      echo_audio=echo(audio, nb_echos)
      return echo_audio

    # Audio effect of missing data
    elif augmentation_type=='corruption':
       drop_rate= kwargs.get('drop_rate',0.01)
       corrupted_audio = corruption(audio, drop_rate)
       return corrupted_audio
    # Audio effect of changing volume
    elif augmentation_type=='volume_change':
        factor= kwargs.get('factor',0.5)
        changed_volume_audio=volume_change(audio, factor)
        return changed_volume_audio

    else:
        raise ValueError(f"Unknown augmentation type: {augmentation_type}")

### 4.2 For the quantizer E1

In [81]:
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained("facebook/hubert-base-ls960")
model = HubertModel.from_pretrained("facebook/hubert-base-ls960", output_hidden_states=True)
model.eval()

HubertModel(
  (feature_extractor): HubertFeatureEncoder(
    (conv_layers): ModuleList(
      (0): HubertGroupNormConvLayer(
        (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,), bias=False)
        (activation): GELUActivation()
        (layer_norm): GroupNorm(512, 512, eps=1e-05, affine=True)
      )
      (1-4): 4 x HubertNoLayerNormConvLayer(
        (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
        (activation): GELUActivation()
      )
      (5-6): 2 x HubertNoLayerNormConvLayer(
        (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
        (activation): GELUActivation()
      )
    )
  )
  (feature_projection): HubertFeatureProjection(
    (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (projection): Linear(in_features=512, out_features=768, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): HubertEncoder(
    (pos_conv_embed): HubertPositionalConvEmbedding(
      (conv): Para

In [82]:
# comment if you do not get this warning anyway
import warnings
warnings.filterwarnings("ignore", message="KMeans is known to have a memory leak on Windows with MKL")

# Make test dataset
kwargs_list = [
    {'augmentation_type' : "gaussian_noise", 'noise_level' : 0.01},
    {'augmentation_type' : "time_stretch", 'rate' : 1.2},
    {'augmentation_type' : "pitch_shift", 'n_steps' : 2},
    {'augmentation_type' : "clipping", 'clip_factor' : 0.8},
    {'augmentation_type' : "lowpass", 'freqlim' : 1000},
    {'augmentation_type' : "bandpass", 'freqlim1' : 500, 'freqlim2' : 1500},
    {'augmentation_type' : "highpass", 'freqlim' : 500},
    {'augmentation_type' : "little_bips"},
    {'augmentation_type' : "big_bips"},
    {'augmentation_type' : "echo", 'nb_echos' : 5},
    {'augmentation_type' : "corruption", 'drop_rate' : 0.01},
    {'augmentation_type' : "volume_change", 'factor' : 0.5}
]
ued_list0 = []
ued_list1 = []
num_samples = 50
vocab_size = 50
features = []
perturbed_features=[]
indices = []
aug_id = 0
# construct a new perturbed dataset with all the augmentations
os.makedirs('features', exist_ok=True)
for kwargs in kwargs_list:
    aug = kwargs.get('augmentation_type')
    audio = [dataset[int(aug_id*num_samples//10) + i] for i in range(num_samples//10)]
    augmentation_fn = lambda x, sampling_rate: augment_audio(x, sr=sampling_rate, **kwargs)
    file_name = "features/"+aug+".pt"
    print("Augmentation", aug_id, ":", file_name)
    if not os.path.exists(file_name):
        preprocess_and_save_features(audio, model, feature_extractor, augmentation_fn, sampling_rate, num_samples=num_samples//10, save_path=file_name)
    features = features + torch.load(file_name, weights_only=False)
    perturbed_features = perturbed_features+torch.load(file_name.replace(".pt", "_perturbed.pt"), weights_only=False)
    features = [torch.tensor(f) if isinstance(f, np.ndarray) else f for f in features] # If numpy array then convert to torch tensor
    perturbed_features = [torch.tensor(f) if isinstance(f, np.ndarray) else f for f in perturbed_features] # If numpy array then convert to torch tensor
    aug_id += 1

# load E0
super_kmeans = KMeans(n_clusters=vocab_size, random_state=42, n_init=10)
fitting_features = torch.cat(features, dim=0)
super_kmeans.fit(fitting_features.numpy())
E0 = lambda x: Quantizer0(x, super_kmeans)

if not os.path.exists('E1_augmented.pth'):
    E1 = MLPQuantizer(input_dim=768, vocab_size=vocab_size)
    E1 = train_quantizer(E0, E1, features, perturbed_features, num_epochs=100, batch_size=5, learning_rate=1e-3)
    # save model E1
    torch.save(E1.state_dict(), "E1_augmented.pth")
else :
    # load E1
    E1 = MLPQuantizer(input_dim=768, vocab_size=vocab_size)
    E1.load_state_dict(torch.load("E1_augmented.pth"))

num_samples = len(features)
for kwargs in kwargs_list:
    test_dataset = [dataset[num_samples + idx] for idx in range(10)]
    augmentation_fn = lambda x, sampling_rate: augment_audio(x, sr=sampling_rate, **kwargs)
    file_name = "features/test"+kwargs.get('augmentation_type')+".pt"
    if not os.path.exists(file_name):
        preprocess_and_save_features(test_dataset, model, feature_extractor, augmentation_fn, sampling_rate, num_samples=10, save_path=file_name)

    # Load precomputed features
    test_features = torch.load(file_name, weights_only=False)
    test_perturbed_features = torch.load(file_name.replace(".pt", "_perturbed.pt"), weights_only=False)

    test_features = [torch.tensor(f) if isinstance(f, np.ndarray) else f for f in test_features] # If numpy array then convert to torch tensor
    test_perturbed_features = [torch.tensor(f) if isinstance(f, np.ndarray) else f for f in test_perturbed_features] # If numpy array then convert to torch tensor
    # compute ued E0:
    E0_fn = lambda x: Quantizer0(x, super_kmeans)
    E0 = E0_quantizer(E0_fn)
    ued0 = compute_ued(test_features, test_perturbed_features, E0)
    print("UED for "+kwargs.get('augmentation_type')+", E0:", ued0)
    ued_list0.append((ued0, kwargs.get('augmentation_type')))

    # compute ued E1:
    ued1 = compute_ued(test_features, test_perturbed_features, E1)
    print("UED for "+kwargs.get('augmentation_type') + ', E1:', ued1)
    ued_list1.append((ued1, kwargs.get('augmentation_type')))

Augmentation 0 : features/gaussian_noise.pt


Processing dataset: 100%|██████████| 5/5 [00:58<00:00, 11.66s/it]


Precomputed features saved at features/gaussian_noise.pt
Augmentation 1 : features/time_stretch.pt


Processing dataset: 100%|██████████| 5/5 [00:33<00:00,  6.80s/it]


Precomputed features saved at features/time_stretch.pt
Augmentation 2 : features/pitch_shift.pt


...

Precomputed features saved at features/testecho.pt
UED for echo, E0: 3.1057240331669305
UED for echo, E1: 5.379930924698726


Processing dataset: 100%|██████████| 10/10 [00:40<00:00,  4.02s/it]


Precomputed features saved at features/testcorruption.pt
UED for corruption, E0: 0.9252887502728602
UED for corruption, E1: 2.212768136421387


Processing dataset: 100%|██████████| 10/10 [00:40<00:00,  4.03s/it]


Precomputed features saved at features/testvolume_change.pt
UED for volume_change, E0: 0.0
UED for volume_change, E1: 0.0


### 4.3 For the quantizer E2

In [83]:
# train quantizer for a second iteration
vocab_size = 50
E1 = MLPQuantizer(input_dim=768, vocab_size=50)
E1.load_state_dict(torch.load("E1_augmented.pth"))
E1_predict = lambda x : E1.predict(x)
if not os.path.exists('E2.pth'):
    # Load quantizer E2
    E2 = MLPQuantizer(input_dim=768, vocab_size=50)
    E2 = train_quantizer(E1_predict, E2, features, perturbed_features, num_epochs=100, batch_size=5, learning_rate=1e-3)
    # save model E2
    torch.save(E2.state_dict(), "E2.pth")
else :
    E2 = MLPQuantizer(input_dim=768, vocab_size=50)
    E2.load_state_dict(torch.load("E2.pth"))

E1 = MLPQuantizer(input_dim=768, vocab_size=50)
E1.load_state_dict(torch.load("E1_augmented.pth"))

ued_list2 = []
num_samples = len(features)
for kwargs in kwargs_list:
    test_dataset = [dataset[num_samples + idx] for idx in range(10)]
    augmentation_fn = lambda x, sampling_rate: augment_audio(x, sr=sampling_rate, **kwargs)
    file_name = "features/test"+kwargs.get('augmentation_type')+".pt"
    if not os.path.exists(file_name):
        preprocess_and_save_features(test_dataset, model, feature_extractor, augmentation_fn, sampling_rate, num_samples=10, save_path=file_name)

    # Load precomputed features
    test_features = torch.load(file_name, weights_only=False)
    test_perturbed_features = torch.load(file_name.replace(".pt", "_perturbed.pt"), weights_only=False)

    test_features = [torch.tensor(f) if isinstance(f, np.ndarray) else f for f in test_features] # If numpy array then convert to torch tensor
    test_perturbed_features = [torch.tensor(f) if isinstance(f, np.ndarray) else f for f in test_perturbed_features] # If numpy array then convert to torch tensor
    # compute ued E1:
    ued1 = compute_ued(test_features, test_perturbed_features, E1)
    print("UED for "+kwargs.get('augmentation_type')+", E1:", ued1)

    # compute ued E2:
    ued2 = compute_ued(test_features, test_perturbed_features, E2)
    print("UED for "+kwargs.get('augmentation_type')+", E2:", ued2)
    ued_list2.append((ued2, kwargs.get('augmentation_type')))

UED for gaussian_noise, E1: 6.223557217612946
UED for gaussian_noise, E2: 5.69997930521107
UED for time_stretch, E1: 4.533992451267993
UED for time_stretch, E2: 4.725976476664051
UED for pitch_shift, E1: 4.086895417855169
UED for pitch_shift, E2: 4.096316757342386
UED for clipping, E1: 10.236287509909802
UED for clipping, E2: 7.701771672378582
UED for lowpass, E1: 9.948334487653375
UED for lowpass, E2: 8.174756371512059
UED for bandpass, E1: 12.859896168750659
UED for bandpass, E2: 8.018711687679835
UED for highpass, E1: 12.548987691464474
UED for highpass, E2: 4.7834242048352635
UED for little_bips, E1: 6.683701332741579
UED for little_bips, E2: 7.726258826919438
UED for big_bips, E1: 3.5981980660618427
UED for big_bips, E2: 2.225251765146783
UED for echo, E1: 5.379930924698726
UED for echo, E2: 5.238284347460697
UED for corruption, E1: 2.212768136421387
UED for corruption, E2: 2.2659410267112468
UED for volume_change, E1: 0.0
UED for volume_change, E2: 0.0


In [84]:
for i, (ued0, ued1, ued2) in enumerate(zip(ued_list0, ued_list1, ued_list2)):
    print(f"Augmentation {i+1}: {ued0[1]}")
    print(f"UED E0: {ued0[0]:.2f}, UED E1: {ued1[0]:.2f}")
    print(f"UED E2: {ued2[0]:.2f}")

Augmentation 1: gaussian_noise
UED E0: 3.04, UED E1: 6.22
UED E2: 5.70
Augmentation 2: time_stretch
UED E0: 2.00, UED E1: 4.53
UED E2: 4.73
Augmentation 3: pitch_shift
UED E0: 1.85, UED E1: 4.09
UED E2: 4.10
Augmentation 4: clipping
UED E0: 6.24, UED E1: 10.24
UED E2: 7.70
Augmentation 5: lowpass
UED E0: 6.07, UED E1: 9.95
UED E2: 8.17
Augmentation 6: bandpass
UED E0: 7.37, UED E1: 12.86
UED E2: 8.02
Augmentation 7: highpass
UED E0: 2.22, UED E1: 12.55
UED E2: 4.78
Augmentation 8: little_bips
UED E0: 5.21, UED E1: 6.68
UED E2: 7.73
Augmentation 9: big_bips
UED E0: 0.77, UED E1: 3.60
UED E2: 2.23
Augmentation 10: echo
UED E0: 3.11, UED E1: 5.38
UED E2: 5.24
Augmentation 11: corruption
UED E0: 0.93, UED E1: 2.21
UED E2: 2.27
Augmentation 12: volume_change
UED E0: 0.00, UED E1: 0.00
UED E2: 0.00


### 4.4 Computation of the ABX score

In [85]:
full_model_0 = FullModel(feature_extractor, model, E0, sampling_rate)
full_model_1 = FullModel(feature_extractor, model, E1, sampling_rate)
full_model_2 = FullModel(feature_extractor, model, E2, sampling_rate)

num_samples = 50

# the next 2 lines would need a different training for kmeans
# abx_error_rate = compute_abx_score(triplets, full_model_0)
# print("ABX error rate for the pretrained quantizer:", abx_error_rate)
abx_error_rate = compute_abx_score(triplets, full_model_1)
print("ABX error rate for the trained quantizer, first iteration:", abx_error_rate)
abx_error_rate = compute_abx_score(triplets, full_model_2)
print("ABX error rate for the second trained quantizer, second iteration:", abx_error_rate)

ABX error rate for the trained quantizer, first iteration: 0.41
ABX error rate for the second trained quantizer, second iteration: 0.32


## 5. Loading and testing new models

In [ ]:
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained("facebook/hubert-large-ls960-ft")
model = HubertModel.from_pretrained("facebook/hubert-large-ls960-ft", output_hidden_states=True)
model.eval()

HubertModel(
  (feature_extractor): HubertFeatureEncoder(
    (conv_layers): ModuleList(
      (0): HubertLayerNormConvLayer(
        (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,))
        (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (activation): GELUActivation()
      )
      (1-4): 4 x HubertLayerNormConvLayer(
        (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,))
        (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (activation): GELUActivation()
      )
      (5-6): 2 x HubertLayerNormConvLayer(
        (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,))
        (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (activation): GELUActivation()
      )
    )
  )
  (feature_projection): HubertFeatureProjection(
    (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (projection): Linear(in_features=512, out_features=1024, bias=True)
    (dropout): Dropout(p=

In [ ]:
# C'est un dataset similaire à celui qu'on veut mais plus petit
# Tous les datasets sont sur HuggingFace de toute façon donc c'est facile à changer
dataset = load_dataset("hf-internal-testing/librispeech_asr_demo", "clean", split="validation", trust_remote_code=True)
dataset = dataset.sort("id")
sampling_rate = dataset.features["audio"].sampling_rate

In [ ]:
# comment if you do not get this warning anyway
import warnings
warnings.filterwarnings("ignore", message="KMeans is known to have a memory leak on Windows with MKL")

# Make test dataset
kwargs_list = [
    {'augmentation_type' : "gaussian_noise", 'noise_level' : 0.01},
    {'augmentation_type' : "time_stretch", 'rate' : 1.2},
    {'augmentation_type' : "pitch_shift", 'n_steps' : 2},
    {'augmentation_type' : "clipping", 'clip_factor' : 0.8},
    {'augmentation_type' : "lowpass", 'freqlim' : 1000},
    {'augmentation_type' : "bandpass", 'freqlim1' : 500, 'freqlim2' : 1500},
    {'augmentation_type' : "highpass", 'freqlim' : 500},
    {'augmentation_type' : "little_bips"},
    {'augmentation_type' : "big_bips"},
    {'augmentation_type' : "echo", 'nb_echos' : 5},
    {'augmentation_type' : "corruption", 'drop_rate' : 0.01},
    {'augmentation_type' : "volume_change", 'factor' : 0.5}
]
ued_list0 = []
ued_list1 = []
num_samples = 50
vocab_size = 50
features = []
perturbed_features=[]
indices = []
aug_id = 0
# construct a new perturbed dataset with all the augmentations
os.makedirs('features_large_10', exist_ok=True)
for kwargs in kwargs_list:
    aug = kwargs.get('augmentation_type')
    audio = [dataset[int(aug_id*num_samples//10) + i] for i in range(num_samples//10)]
    augmentation_fn = lambda x, sampling_rate: augment_audio(x, sr=sampling_rate, **kwargs)
    file_name = "features_large_10/"+aug+".pt"
    print("Augmentation", aug_id, ":", file_name)
    if not os.path.exists(file_name):
        preprocess_and_save_features(audio, model, feature_extractor, augmentation_fn, sampling_rate, num_samples=num_samples//10, save_path=file_name)
    features = features + torch.load(file_name, weights_only=False)
    perturbed_features = perturbed_features+torch.load(file_name.replace(".pt", "_perturbed.pt"), weights_only=False)
    features = [torch.tensor(f) if isinstance(f, np.ndarray) else f for f in features] # If numpy array then convert to torch tensor
    perturbed_features = [torch.tensor(f) if isinstance(f, np.ndarray) else f for f in perturbed_features] # If numpy array then convert to torch tensor
    aug_id += 1

# load E0
super_kmeans = KMeans(n_clusters=vocab_size, random_state=42, n_init=10)
fitting_features = torch.cat(features, dim=0)
super_kmeans.fit(fitting_features.numpy())
E0 = lambda x: Quantizer0(x, super_kmeans)

if not os.path.exists('E1_augmented_large.pth'):
    E1 = MLPQuantizer(input_dim=1024, vocab_size=vocab_size)
    E1 = train_quantizer(E0, E1, features, perturbed_features, num_epochs=100, batch_size=5, learning_rate=1e-3)
    # save model E1
    torch.save(E1.state_dict(), "E1_augmented_large.pth")
else :
    # load E1
    E1 = MLPQuantizer(input_dim=1024, vocab_size=vocab_size)
    E1.load_state_dict(torch.load("E1_augmented_large.pth"))

num_samples = len(features)
for kwargs in kwargs_list:
    test_dataset = [dataset[num_samples + idx] for idx in range(10)]
    augmentation_fn = lambda x, sampling_rate: augment_audio(x, sr=sampling_rate, **kwargs)
    file_name = "features_large_10/test"+kwargs.get('augmentation_type')+".pt"
    if not os.path.exists(file_name):
        preprocess_and_save_features(test_dataset, model, feature_extractor, augmentation_fn, sampling_rate, num_samples=10, save_path=file_name)

    # Load precomputed features
    test_features = torch.load(file_name, weights_only=False)
    test_perturbed_features = torch.load(file_name.replace(".pt", "_perturbed.pt"), weights_only=False)

    test_features = [torch.tensor(f) if isinstance(f, np.ndarray) else f for f in test_features] # If numpy array then convert to torch tensor
    test_perturbed_features = [torch.tensor(f) if isinstance(f, np.ndarray) else f for f in test_perturbed_features] # If numpy array then convert to torch tensor
    # compute ued E0:
    E0_fn = lambda x: Quantizer0(x, super_kmeans)
    E0 = E0_quantizer(E0_fn)
    ued0 = compute_ued(test_features, test_perturbed_features, E0)
    print("UED for "+kwargs.get('augmentation_type')+", E0:", ued0)
    ued_list0.append((ued0, kwargs.get('augmentation_type')))

    # compute ued E1:
    ued1 = compute_ued(test_features, test_perturbed_features, E1)
    print("UED for "+kwargs.get('augmentation_type') + ', E1:', ued1)
    ued_list1.append((ued1, kwargs.get('augmentation_type')))

Augmentation 0 : features_large_10/gaussian_noise.pt


Processing dataset: 100%|██████████| 5/5 [00:12<00:00,  2.58s/it]


Precomputed features saved at features_large_10/gaussian_noise.pt
Augmentation 1 : features_large_10/time_stretch.pt


Processing dataset: 100%|██████████| 5/5 [00:10<00:00,  2.18s/it]


Precomputed features saved at features_large_10/time_stretch.pt
Augmentation 2 : features_large_10/pitch_shift.pt


...

Precomputed features saved at features_large_10/testecho.pt
UED for echo, E0: 2.528698894059694
UED for echo, E1: 3.1380226011448773


Processing dataset: 100%|██████████| 10/10 [00:13<00:00,  1.37s/it]


Precomputed features saved at features_large_10/testcorruption.pt
UED for corruption, E0: 0.8896540988589654
UED for corruption, E1: 1.758846675797606


Processing dataset: 100%|██████████| 10/10 [00:13<00:00,  1.36s/it]


Precomputed features saved at features_large_10/testvolume_change.pt
UED for volume_change, E0: 0.0
UED for volume_change, E1: 0.0


In [ ]:
# train quantizer for a second iteration
vocab_size = 50
E1 = MLPQuantizer(input_dim=1024, vocab_size=50)
E1.load_state_dict(torch.load("E1_augmented_large.pth"))
E1_predict = lambda x : E1.predict(x)
if not os.path.exists('E2_large.pth'):
    # Load quantizer E2
    E2 = MLPQuantizer(input_dim=1024, vocab_size=50)
    E2 = train_quantizer(E1_predict, E2, features, perturbed_features, num_epochs=100, batch_size=5, learning_rate=1e-3)
    # save model E2
    torch.save(E2.state_dict(), "E2_large.pth")
else :
    E2 = MLPQuantizer(input_dim=1024, vocab_size=50)
    E2.load_state_dict(torch.load("E2_large.pth"))

E1 = MLPQuantizer(input_dim=1024, vocab_size=50)
E1.load_state_dict(torch.load("E1_augmented_large.pth"))

ued_list2 = []
num_samples = len(features)
for kwargs in kwargs_list:
    test_dataset = [dataset[num_samples + idx] for idx in range(10)]
    augmentation_fn = lambda x, sampling_rate: augment_audio(x, sr=sampling_rate, **kwargs)
    file_name = "features_large_10/test2"+kwargs.get('augmentation_type')+".pt"
    if not os.path.exists(file_name):
        preprocess_and_save_features(test_dataset, model, feature_extractor, augmentation_fn, sampling_rate, num_samples=10, save_path=file_name)

    # Load precomputed features
    test_features = torch.load(file_name, weights_only=False)
    test_perturbed_features = torch.load(file_name.replace(".pt", "_perturbed.pt"), weights_only=False)

    test_features = [torch.tensor(f) if isinstance(f, np.ndarray) else f for f in test_features] # If numpy array then convert to torch tensor
    test_perturbed_features = [torch.tensor(f) if isinstance(f, np.ndarray) else f for f in test_perturbed_features] # If numpy array then convert to torch tensor
    # compute ued E1:
    ued1 = compute_ued(test_features, test_perturbed_features, E1)
    print("UED for "+kwargs.get('augmentation_type')+", E1:", ued1)

    # compute ued E2:
    ued2 = compute_ued(test_features, test_perturbed_features, E2)
    print("UED for "+kwargs.get('augmentation_type')+", E2:", ued2)
    ued_list2.append((ued2, kwargs.get('augmentation_type')))

Epoch 1/100 - CTC Loss: 4.3465
Epoch 2/100 - CTC Loss: -0.1237
Epoch 3/100 - CTC Loss: -0.5542
Epoch 4/100 - CTC Loss: -0.6293
Epoch 5/100 - CTC Loss: -0.6618
Epoch 6/100 - CTC Loss: -0.6830
Epoch 7/100 - CTC Loss: -0.6988
Epoch 8/100 - CTC Loss: -0.7101
Epoch 9/100 - CTC Loss: -0.7223
Epoch 10/100 - CTC Loss: -0.7318
Epoch 11/100 - CTC Loss: -0.7390
Epoch 12/100 - CTC Loss: -0.7487
Epoch 13/100 - CTC Loss: -0.7549
Epoch 14/100 - CTC Loss: -0.7641
Epoch 15/100 - CTC Loss: -0.7716
Epoch 16/100 - CTC Loss: -0.7805
Epoch 17/100 - CTC Loss: -0.7872
Epoch 18/100 - CTC Loss: -0.7918
Epoch 19/100 - CTC Loss: -0.8006
Epoch 20/100 - CTC Loss: -0.8070
Epoch 21/100 - CTC Loss: -0.8098
Epoch 22/100 - CTC Loss: -0.8158
Epoch 23/100 - CTC Loss: -0.8257
Epoch 24/100 - CTC Loss: -0.8316
Epoch 25/100 - CTC Loss: -0.8353
Epoch 26/100 - CTC Loss: -0.8422
Epoch 27/100 - CTC Loss: -0.8489
Epoch 28/100 - CTC Loss: -0.8511
Epoch 29/100 - CTC Loss: -0.8570
Epoch 30/100 - CTC Loss: -0.8646
Epoch 31/100 - CTC L

Processing dataset: 100%|██████████| 10/10 [00:13<00:00,  1.36s/it]


Precomputed features saved at features_large_10/test2gaussian_noise.pt
UED for gaussian_noise, E1: 2.7788915631100912
UED for gaussian_noise, E2: 4.35


Processing dataset: 100%|██████████| 10/10 [00:12<00:00,  1.26s/it]


Precomputed features saved at features_large_10/test2time_stretch.pt
UED for time_stretch, E1: 3.1035225010370007
UED for time_stretch, E2: 4.55


...

Precomputed features saved at features_large_10/test2echo.pt
UED for echo, E1: 3.1380226011448773
UED for echo, E2: 3.5


Processing dataset: 100%|██████████| 10/10 [00:13<00:00,  1.36s/it]


Precomputed features saved at features_large_10/test2corruption.pt
UED for corruption, E1: 1.9442652295409089
UED for corruption, E2: 1.1


Processing dataset: 100%|██████████| 10/10 [00:13<00:00,  1.35s/it]


Precomputed features saved at features_large_10/test2volume_change.pt
UED for volume_change, E1: 0.0
UED for volume_change, E2: 0.0


In [ ]:
# train quantizer for a second iteration
vocab_size = 50
E2 = MLPQuantizer(input_dim=1024, vocab_size=50)
E2.load_state_dict(torch.load("E2_large.pth"))
E2_predict = lambda x : E2.predict(x)
if not os.path.exists('E3_large.pth'):
    # Load quantizer E3
    E3 = MLPQuantizer(input_dim=1024, vocab_size=50)
    E3 = train_quantizer(E2_predict, E3, features, perturbed_features, num_epochs=100, batch_size=5, learning_rate=1e-3)
    # save model E3
    torch.save(E3.state_dict(), "E3_large.pth")
else :
    E3 = MLPQuantizer(input_dim=1024, vocab_size=50)
    E3.load_state_dict(torch.load("E3_large.pth"))

E2 = MLPQuantizer(input_dim=1024, vocab_size=50)
E2.load_state_dict(torch.load("E2_large.pth"))

ued_list3 = []
num_samples = len(features)
for kwargs in kwargs_list:
    test_dataset = [dataset[num_samples + idx] for idx in range(10)]
    augmentation_fn = lambda x, sampling_rate: augment_audio(x, sr=sampling_rate, **kwargs)
    file_name = "features_large_10/test3"+kwargs.get('augmentation_type')+".pt"
    preprocess_and_save_features(test_dataset, model, feature_extractor, augmentation_fn, sampling_rate, num_samples=10, save_path=file_name)

    # Load precomputed features
    test_features = torch.load(file_name, weights_only=False)
    test_perturbed_features = torch.load(file_name.replace(".pt", "_perturbed.pt"), weights_only=False)

    test_features = [torch.tensor(f) if isinstance(f, np.ndarray) else f for f in test_features] # If numpy array then convert to torch tensor
    test_perturbed_features = [torch.tensor(f) if isinstance(f, np.ndarray) else f for f in test_perturbed_features] # If numpy array then convert to torch tensor
    # compute ued E2:
    ued2 = compute_ued(test_features, test_perturbed_features, E2)
    print("UED for "+kwargs.get('augmentation_type')+", E2:", ued2)

    # compute ued E3:
    ued3 = compute_ued(test_features, test_perturbed_features, E3)
    print("UED for "+kwargs.get('augmentation_type')+", E3:", ued3)
    ued_list3.append((ued3, kwargs.get('augmentation_type')))

Epoch 1/100 - CTC Loss: 4.8647
Epoch 2/100 - CTC Loss: -0.5271
Epoch 3/100 - CTC Loss: -1.1823
Epoch 4/100 - CTC Loss: -1.2951
Epoch 5/100 - CTC Loss: -1.3231
Epoch 6/100 - CTC Loss: -1.3392
Epoch 7/100 - CTC Loss: -1.3485
Epoch 8/100 - CTC Loss: -1.3559
Epoch 9/100 - CTC Loss: -1.3609
Epoch 10/100 - CTC Loss: -1.3654
Epoch 11/100 - CTC Loss: -1.3703
Epoch 12/100 - CTC Loss: -1.3739
Epoch 13/100 - CTC Loss: -1.3766
Epoch 14/100 - CTC Loss: -1.3804
Epoch 15/100 - CTC Loss: -1.3830
Epoch 16/100 - CTC Loss: -1.3857
Epoch 17/100 - CTC Loss: -1.3880
Epoch 18/100 - CTC Loss: -1.3902
Epoch 19/100 - CTC Loss: -1.3922
Epoch 20/100 - CTC Loss: -1.3941
Epoch 21/100 - CTC Loss: -1.3959
Epoch 22/100 - CTC Loss: -1.3973
Epoch 23/100 - CTC Loss: -1.3988
Epoch 24/100 - CTC Loss: -1.4007
Epoch 25/100 - CTC Loss: -1.4020
Epoch 26/100 - CTC Loss: -1.4031
Epoch 27/100 - CTC Loss: -1.4048
Epoch 28/100 - CTC Loss: -1.4058
Epoch 29/100 - CTC Loss: -1.4064
Epoch 30/100 - CTC Loss: -1.4084
Epoch 31/100 - CTC L

Processing dataset: 100%|██████████| 10/10 [00:13<00:00,  1.34s/it]


Precomputed features saved at features_large_10/test3gaussian_noise.pt
UED for gaussian_noise, E2: 4.1
UED for gaussian_noise, E3: 0.0


Processing dataset: 100%|██████████| 10/10 [00:12<00:00,  1.26s/it]


Precomputed features saved at features_large_10/test3time_stretch.pt
UED for time_stretch, E2: 4.55
UED for time_stretch, E3: 0.0


...

Precomputed features saved at features_large_10/test3echo.pt
UED for echo, E2: 3.5
UED for echo, E3: 0.0


Processing dataset: 100%|██████████| 10/10 [00:13<00:00,  1.37s/it]


Precomputed features saved at features_large_10/test3corruption.pt
UED for corruption, E2: 1.3
UED for corruption, E3: 1.0


Processing dataset: 100%|██████████| 10/10 [00:13<00:00,  1.39s/it]


Precomputed features saved at features_large_10/test3volume_change.pt
UED for volume_change, E2: 0.0
UED for volume_change, E3: 0.0


In [ ]:
full_model_0 = FullModel(feature_extractor, model, E0, sampling_rate)
full_model_1 = FullModel(feature_extractor, model, E1, sampling_rate)
full_model_2 = FullModel(feature_extractor, model, E2, sampling_rate)
full_model_3 = FullModel(feature_extractor, model, E3, sampling_rate)

num_samples = 50

# the next 2 lines would need a different training for kmeans
# abx_error_rate = compute_abx_score(triplets, full_model_0)
# print("ABX error rate for the pretrained quantizer:", abx_error_rate)
abx_error_rate = compute_abx_score(triplets, full_model_1)
print("ABX error rate for the trained quantizer, first iteration:", abx_error_rate)
abx_error_rate = compute_abx_score(triplets, full_model_2)
print("ABX error rate for the second trained quantizer, second iteration:", abx_error_rate)
abx_error_rate = compute_abx_score(triplets, full_model_3)
print("ABX error rate for the second trained quantizer, second iteration:", abx_error_rate)

ABX error rate for the trained quantizer, first iteration: 0.32
ABX error rate for the second trained quantizer, second iteration: 0.33
ABX error rate for the second trained quantizer, second iteration: 0.33


## 6. Dynamic Time Warping

In [36]:
from dtaidistance import dtw_ndim

In [92]:
## You can always use this routine before anything to get the preprocessed datasets and your E0 super kmeans.
# load dataset
num_samples = 10
vocab_size = 50
augmentation_fn = lambda x, sampling_rate: augment_audio(x, sr=sampling_rate, augmentation_type="gaussian_noise", noise_level=0.01)
file_name = "precomputed_features.pt"
if not os.path.exists(file_name):
    preprocess_and_save_features(dataset, model, feature_extractor, augmentation_fn, sampling_rate, num_samples=num_samples, save_path=file_name)

# Load precomputed features
features = torch.load(file_name, weights_only=False)
perturbed_features = torch.load(file_name.replace(".pt", "_perturbed.pt"), weights_only=False)

features = [torch.tensor(f) if isinstance(f, np.ndarray) else f for f in features] # If numpy array then convert to torch tensor
perturbed_features = [torch.tensor(f) if isinstance(f, np.ndarray) else f for f in perturbed_features] # If numpy array then convert to torch tensor

# FIT a super kmeans on all data that will be used for E0
super_kmeans = KMeans(n_clusters=vocab_size, random_state=42, n_init=10)
fitting_features = torch.cat(features, dim=0)
super_kmeans.fit(fitting_features.numpy())

KMeans(n_clusters=50, n_init=10, random_state=42)

### 6.1 The function

In [93]:
def compute_dtw(dataset, perturbed_dataset, quantizer, embeddings):
    # quantizer sur le dataset et le perturbed_dataset, puis comparer en utilisant le dtw, en utilisant les embeddings pour avoir une représentation dense
    # pour cela on utiliserait normalement plutôt les embeddings de l'unit-to-speech
    # comme ils ne sont pas disponibles, je vais plutôt utiliser les centroïdes du KMeans ? c'est moins réaliste mais ça donne déjà une idée
    total_distance = 0

    for (x, augmented_x) in tqdm(zip(dataset, perturbed_dataset)):
        x = x.unsqueeze(0)
        augmented_x = augmented_x.unsqueeze(0)
        quantized_x = np.array(
            deduplicate(quantizer.predict(x).flatten().tolist())
        ) # Convert to token sequence
        quantized_aug_x = np.array(
            deduplicate(quantizer.predict(augmented_x).flatten().tolist())
        )
        # Compute dtw
        embedded_quantized_x = embeddings[quantized_x]
        embedded_quantized_aug_x = embeddings[quantized_aug_x]
        dtw_distance = dtw_ndim.distance(embedded_quantized_x, embedded_quantized_aug_x) # /!\ in a mathematical sense, dtw is not a distance
        total_distance += dtw_distance / len(quantized_x)

    return total_distance

### 6.2 Tests

In [94]:
# compute ued and dtw E0:
E0_fn = lambda x: Quantizer0(x, super_kmeans)
E0 = E0_quantizer(E0_fn)
ued = compute_ued(test_features, test_perturbed_features, E0)
dtw_distance = compute_dtw(features, perturbed_features, E0, super_kmeans.cluster_centers_)
print("UED:", ued)
print("dtw", dtw_distance)

# compute ued and dtw E1:
ued = compute_ued(test_features, test_perturbed_features, E1)
dtw_distance = compute_dtw(features, perturbed_features, E1, super_kmeans.cluster_centers_)
print("UED:", ued)
print("dtw", dtw_distance)

10it [00:11,  1.17s/it]


UED: 0.0
dtw 3.7972295699352476


10it [00:12,  1.25s/it]

UED: 0.0
dtw 4.476936923988386


## 7. Quantizer architecture

In this section we will try to improve the results obtained before as we did not observe much change. The question we are trying to adress is about our quantizer robustness. There exist different architecture known for improving robustness but the most used one is obviously the attention based architecture (like Transformers, GAT, etc). We are currently studying a signal processing model, thus, we want our signal to be able to tell wich part of the input is important in order to tokenize some other given part of the signal.

We will use an attention mechanism here because it is the most famous kind but obviously, process like LSTM and RNN could have been a good beginning.

### 7.0 Some set up

In [37]:
# Load feature extractor and model
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained("facebook/wav2vec2-base")
model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base")

/usr/local/lib/python3.11/dist-packages/transformers/configuration_utils.py:315: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(


In [38]:
# Make test dataset
kwargs_list = [
    {'augmentation_type' : "gaussian_noise", 'noise_level' : 0.01},
    {'augmentation_type' : "time_stretch", 'rate' : 1.2},
    {'augmentation_type' : "pitch_shift", 'n_steps' : 2},
    {'augmentation_type' : "clipping", 'clip_factor' : 0.8},
    {'augmentation_type' : "lowpass", 'freqlim' : 1000},
    {'augmentation_type' : "bandpass", 'freqlim1' : 500, 'freqlim2' : 1500},
    {'augmentation_type' : "highpass", 'freqlim' : 500},
    {'augmentation_type' : "little_bips"},
    {'augmentation_type' : "big_bips"},
    {'augmentation_type' : "echo", 'nb_echos' : 5},
    {'augmentation_type' : "corruption", 'drop_rate' : 0.01},
    {'augmentation_type' : "volume_change", 'factor' : 0.5}
]
num_samples = 50
vocab_size = 50
features = []
perturbed_features=[]
indices = []
aug_id = 0
# construct a new perturbed dataset with all the augmentations
os.makedirs('features', exist_ok=True)
for kwargs in kwargs_list:
    aug = kwargs.get('augmentation_type')
    audio = [dataset[int(aug_id*num_samples//10) + i] for i in range(num_samples//10)]
    augmentation_fn = lambda x, sampling_rate: augment_audio(x, sr=sampling_rate, **kwargs)
    file_name = "features/"+aug+".pt"
    print("Augmentation", aug_id, ":", file_name)
    if not os.path.exists(file_name):
        preprocess_and_save_features(audio, model, feature_extractor, augmentation_fn, sampling_rate, num_samples=num_samples//10, save_path=file_name)
    features = features + torch.load(file_name, weights_only=False)
    perturbed_features = perturbed_features+torch.load(file_name.replace(".pt", "_perturbed.pt"), weights_only=False)
    features = [torch.tensor(f) if isinstance(f, np.ndarray) else f for f in features] # If numpy array then convert to torch tensor
    perturbed_features = [torch.tensor(f) if isinstance(f, np.ndarray) else f for f in perturbed_features] # If numpy array then convert to torch tensor
    aug_id += 1

# load E0
super_kmeans = KMeans(n_clusters=vocab_size, random_state=42, n_init=10)
fitting_features = torch.cat(features, dim=0)
super_kmeans.fit(fitting_features.numpy())
E0 = lambda x: Quantizer0(x, super_kmeans)

Augmentation 0 : features/gaussian_noise.pt
Augmentation 1 : features/time_stretch.pt
Augmentation 2 : features/pitch_shift.pt
Augmentation 3 : features/clipping.pt
Augmentation 4 : features/lowpass.pt
Augmentation 5 : features/bandpass.pt
Augmentation 6 : features/highpass.pt
Augmentation 7 : features/little_bips.pt
Augmentation 8 : features/big_bips.pt
Augmentation 9 : features/echo.pt
Augmentation 10 : features/corruption.pt
Augmentation 11 : features/volume_change.pt


### 7.1 Some new quantizer

In [40]:
class BoostedMLPQuantizer(MLPQuantizer):
    """Trainable quantizer E1 that uses a MLP of same number of parameter than TransformerQuantizer."""
    def __init__(self, input_dim: int, vocab_size: int, num_parameters: int, num_layers =3):
        assert num_layers > 2, "num_layers must be greater than 2 to have hidden layers and attain the same number of parameters without uselessly huge matrices"
        n_inter = np.sqrt(4*num_parameters*(num_layers-2)+(input_dim+vocab_size)**2)-(input_dim+vocab_size)
        n_intermediaire_parameter = int(n_inter/2*(num_layers-2))
        hidden_dim = [n_intermediaire_parameter] * (num_layers-1)
        super().__init__(input_dim, vocab_size, hidden_dim = hidden_dim)

In [115]:
class TransformerQuantizer(nn.Module):
    def __init__(self, input_dim, vocab_size, max_seq_len=512, num_heads=1, num_layers=1, hidden_dim=368):
        super().__init__()
        self.input_dim = input_dim
        self.vocab_size = vocab_size
        self.max_seq_len = max_seq_len
        self.pos_embedding = nn.Parameter(torch.randn(1, max_seq_len, input_dim))
        encoder_layer = nn.TransformerEncoderLayer(d_model=input_dim, nhead=num_heads, dim_feedforward=hidden_dim)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.output_layer = nn.Linear(input_dim, vocab_size)

    def forward(self, features):
        remove_batch = False
        if features.dim() > 3:
            remove_batch = True
            features = features.squeeze(0)
        batch, time_steps, _ = features.size()
        # Pad or truncate positional embedding to match input sequence length
        pos_emb = self.pos_embedding[:, :min(time_steps, self.max_seq_len), :]
        if time_steps > self.max_seq_len:
          pad_len = time_steps - self.max_seq_len
          pos_emb = F.pad(pos_emb, (0, 0, 0, pad_len))
        features = features + pos_emb

        features = features.transpose(0, 1)  # [time, batch, input_dim]
        encoded_features = self.transformer_encoder(features)
        encoded_features = encoded_features.transpose(0, 1)  # [batch, time, input_dim]
        logits = self.output_layer(encoded_features)
        if remove_batch:
          logits = logits.unsqueeze(0)
        return logits

    def predict(self, features):
        if features.dim() > 3:
            features = features.squeeze(0)
        logits = self(features).squeeze(0)
        predictions = logits.argmax(dim=-1)
        return predictions

### 7.2 Experiment

In [58]:
# define E1 in form of a Transformer
E1T = TransformerQuantizer(input_dim=768, vocab_size=vocab_size)
# count the number of parameters
Nparam = sum(p.numel() for p in E1T.parameters())
print("Number of parameters:", Nparam)

Number of parameters: 3363490


In [59]:
# define a boosted MLP with same number of parameter
E1B = BoostedMLPQuantizer(input_dim=768, vocab_size=vocab_size, num_parameters=Nparam)
# count the number of parameters
print("Number of parameters:", sum(p.numel() for p in E1B.parameters()))

Number of parameters: 3366350


#### 7.2.1 Training

I won't do training of E2 and more. One could expect that improving the performance of E1 also lead to an improvement for iteratively trained similar architecture.

In [110]:
E1T = train_quantizer(E0, E1T, features, perturbed_features, num_epochs=40, batch_size=5, learning_rate=1e-3)
E1B = train_quantizer(E0, E1B, features, perturbed_features, num_epochs=40, batch_size=5, learning_rate=1e-3)
# save model E1T & E1B
torch.save(E1T.state_dict(), "E1_transformer.pth")
torch.save(E1B.state_dict(), "E1_boosted_mlp.pth")

#### 7.2.2 Testing

In [116]:
# load E1T and E1B
E1T = TransformerQuantizer(input_dim=768, vocab_size=vocab_size)
E1T.load_state_dict(torch.load("E1_transformer.pth"))
E1B = BoostedMLPQuantizer(input_dim=768, vocab_size=vocab_size, num_parameters=Nparam)
E1B.load_state_dict(torch.load("E1_boosted_mlp.pth"))

<All keys matched successfully>

In [66]:
E1list = [E1T, E1B]
ued_dict = {}
ued_dict["E0"] = []
ued_dict["E1 n"+str(1)] = []
ued_dict["E1 n"+str(2)] = []
for i, E1 in enumerate(E1list):
    print("Quantizer tested : ", i+1)
    num_samples = len(features)
    for kwargs in kwargs_list:
        test_dataset = [dataset[num_samples + idx] for idx in range(10)]
        augmentation_fn = lambda x, sampling_rate: augment_audio(x, sr=sampling_rate, **kwargs)
        file_name = "features/test"+kwargs.get('augmentation_type')+".pt"
        if not os.path.exists(file_name):
            preprocess_and_save_features(test_dataset, model, feature_extractor, augmentation_fn, sampling_rate, num_samples=10, save_path=file_name)

        # Load precomputed features
        test_features = torch.load(file_name, weights_only=False)
        test_perturbed_features = torch.load(file_name.replace(".pt", "_perturbed.pt"), weights_only=False)

        test_features = [torch.tensor(f) if isinstance(f, np.ndarray) else f for f in test_features] # If numpy array then convert to torch tensor
        test_perturbed_features = [torch.tensor(f) if isinstance(f, np.ndarray) else f for f in test_perturbed_features] # If numpy array then convert to torch tensor
        # compute ued E0:
        if i==0:
          E0_fn = lambda x: Quantizer0(x, super_kmeans)
          E0 = E0_quantizer(E0_fn)
          ued0 = compute_ued(test_features, test_perturbed_features, E0)
          print("UED for "+kwargs.get('augmentation_type')+", E0:", ued0)
          ued_dict["E0"].append((ued0, kwargs.get('augmentation_type')))

        # compute ued E1:
        ued1 = compute_ued(test_features, test_perturbed_features, E1)
        print("UED for "+kwargs.get('augmentation_type') + ', E1:', ued1)
        ued_dict["E1 n"+str(i+1)].append((ued1, kwargs.get('augmentation_type')))

Quantizer tested :  1
UED for gaussian_noise, E0: 3.038964608924337
UED for gaussian_noise, E1: 3.36651585215187
UED for time_stretch, E0: 2.0019564018167113
UED for time_stretch, E1: 4.4591887364666185
UED for pitch_shift, E0: 1.8452420917622494
UED for pitch_shift, E1: 3.989184164816373
UED for clipping, E0: 6.241339214482558
UED for clipping, E1: 3.7098757235437647
UED for lowpass, E0: 6.071338600328875
UED for lowpass, E1: 4.048818559238496
UED for bandpass, E0: 7.370281839454777
UED for bandpass, E1: 4.239444293936026
UED for highpass, E0: 2.2220905701114693
UED for highpass, E1: 3.0259689981000055
UED for little_bips, E0: 5.2145146137484275
UED for little_bips, E1: 3.9973782165460614
UED for big_bips, E0: 0.7681925187432908
UED for big_bips, E1: 2.418035795665977
UED for echo, E0: 3.1057240331669305
UED for echo, E1: 3.330847007173681
UED for corruption, E0: 0.9252887502728602
UED for corruption, E1: 2.3952010684069847
UED for volume_change, E0: 0.0
UED for volume_change, E1: 2.2

In [117]:
full_model_1 = FullModel(feature_extractor, model, E1T, sampling_rate)
full_model_2 = FullModel(feature_extractor, model, E1B, sampling_rate)

num_samples = 50

ABX_score(full_model_1)
ABX_score(full_model_2)

Resolving data files:   0%|          | 0/49 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/66 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/49 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/66 [00:00<?, ?it/s]

Sampling rate: 16000
Created 100 ABX triplets.
ABX error rate: 0.37


Resolving data files:   0%|          | 0/49 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/66 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/49 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/66 [00:00<?, ?it/s]

Sampling rate: 16000
Created 100 ABX triplets.
ABX error rate: 0.33
